In [30]:
# ============================================================
# CELL 1
# State constructors for Gaussian / non-Gaussian simulator
#
# Physics convention
# ------------------
# Fock basis:
#
#     |0>, |1>, |2>, ...
#
# Coherent state:
#
#             infinity
#     |a> = exp(-|a|^2/2) sum a^n/sqrt(n!) |n>
#               n=0
#
# Squeezed vacuum:
#
#     |0,z> = 1/sqrt(cosh r)
#             sum_k [
#               (-exp(i phi) tanh r)^k
#               sqrt((2k)!)/(2^k k!)
#             ] |2k>
#
# with z = r exp(i phi).
#
# Mixture:
#
#     rho(p) = (1-p)|0><0| + p|1><1|
#
#
# IMPORTANT
# ---------
# "dim" is the Hilbert-space dimension used internally.
#
# "max_n" is the highest Fock state deliberately retained.
#
# Therefore:
#
#   large dim + max_n < dim-1
#       deliberate Fock truncation experiment
#
#   large dim + max_n = None
#       best finite-dimensional approximation to the
#       ideal coherent / squeezed state.
#
# Tests are included at the bottom of this cell.
# ============================================================


import math
import numpy as np

from qutip import Qobj, basis, ket2dm


# ============================================================
# VALIDATION HELPERS
# ============================================================

def _require_dimension(dim):
    """
    Require a positive integer Hilbert-space dimension.
    """
    if not isinstance(dim, (int, np.integer)):
        raise TypeError(
            "dim must be an integer."
        )

    if dim < 1:
        raise ValueError(
            "dim must be >= 1."
        )

    return int(dim)


def _require_max_n(max_n, dim):
    """
    Validate highest retained Fock state.
    """
    if max_n is None:
        return dim - 1

    if not isinstance(
        max_n,
        (int, np.integer),
    ):
        raise TypeError(
            "max_n must be an integer or None."
        )

    if max_n < 0:
        raise ValueError(
            "max_n must be >= 0."
        )

    if max_n >= dim:
        raise ValueError(
            "max_n must satisfy max_n < dim."
        )

    return int(max_n)


def _normalized_ket_from_coefficients(
    coefficients,
):
    """
    Convert coefficient vector to a normalized QuTiP ket.
    """
    coefficients = np.asarray(
        coefficients,
        dtype=complex,
    )

    norm = np.sqrt(
        np.sum(
            np.abs(coefficients) ** 2
        )
    )

    if norm <= 0.0:
        raise ValueError(
            "State coefficients have zero norm."
        )

    coefficients = (
        coefficients / norm
    )

    return Qobj(
        coefficients.reshape(-1, 1)
    )


# ============================================================
# FOCK STATE
# ============================================================

def fock_state(
    dim,
    n,
):
    """
    Return |n> in a Hilbert space of dimension dim.
    """
    dim = _require_dimension(
        dim
    )

    if not isinstance(
        n,
        (int, np.integer),
    ):
        raise TypeError(
            "n must be an integer."
        )

    if n < 0 or n >= dim:
        raise ValueError(
            "Require 0 <= n < dim."
        )

    return basis(
        dim,
        int(n),
    )


# ============================================================
# COHERENT-STATE COEFFICIENTS
# ============================================================

def coherent_fock_coefficients(
    dim,
    alpha,
    max_n=None,
):
    """
    Return the UNRENORMALIZED analytic coherent-state
    coefficients retained in the finite Fock basis.

    Coefficients satisfy

        c_0 = exp(-|alpha|^2/2)

        c_(n+1) = alpha/sqrt(n+1) * c_n

    States above max_n are set to zero.
    """
    dim = _require_dimension(
        dim
    )

    max_n = _require_max_n(
        max_n,
        dim,
    )

    alpha = complex(
        alpha
    )

    c = np.zeros(
        dim,
        dtype=complex,
    )

    c[0] = np.exp(
        -0.5
        * abs(alpha) ** 2
    )

    for n in range(max_n):

        c[n + 1] = (
            c[n]
            * alpha
            / np.sqrt(n + 1)
        )

    return c


def coherent_retained_probability(
    dim,
    alpha,
    max_n=None,
):
    """
    Probability contained in the retained coherent-state
    Fock components BEFORE numerical renormalization.

    For sufficiently large max_n this approaches 1.
    """
    c = coherent_fock_coefficients(
        dim,
        alpha,
        max_n=max_n,
    )

    return float(
        np.sum(
            np.abs(c) ** 2
        )
    )


def coherent_state(
    dim,
    alpha,
    max_n=None,
):
    """
    Return normalized finite-Fock representation
    of a coherent state.

    max_n=None:
        retain all states available in dim.

    max_n < dim-1:
        deliberately truncated coherent state.
    """
    c = coherent_fock_coefficients(
        dim,
        alpha,
        max_n=max_n,
    )

    return (
        _normalized_ket_from_coefficients(
            c
        )
    )


# ============================================================
# SQUEEZED-VACUUM COEFFICIENTS
# ============================================================

def squeezed_vacuum_fock_coefficients(
    dim,
    r,
    phi=0.0,
    max_n=None,
):
    """
    Return UNRENORMALIZED squeezed-vacuum coefficients.

    Convention:

        z = r exp(i phi)

    and

        |0,z> =
            1/sqrt(cosh r)
            sum_k [
              (-exp(i phi) tanh r)^k
              sqrt((2k)!)/(2^k k!)
            ] |2k>

    Only even Fock states are populated.

    A recurrence is used to avoid factorial overflow.
    """
    dim = _require_dimension(
        dim
    )

    max_n = _require_max_n(
        max_n,
        dim,
    )

    r = float(r)
    phi = float(phi)

    if r < 0.0:
        raise ValueError(
            "r must be >= 0."
        )

    c = np.zeros(
        dim,
        dtype=complex,
    )

    # Vacuum coefficient.
    c[0] = (
        1.0
        / np.sqrt(
            np.cosh(r)
        )
    )

    factor = (
        -np.exp(1j * phi)
        * np.tanh(r)
    )

    # k labels Fock pairs:
    #
    # k = 0 -> |0>
    # k = 1 -> |2>
    # k = 2 -> |4>
    # ...

    k = 0

    while True:

        n_current = 2 * k
        n_next = n_current + 2

        if n_next > max_n:
            break

        if n_next >= dim:
            break

        ratio = (
            factor
            * np.sqrt(
                (2 * k + 2)
                * (2 * k + 1)
            )
            / (
                2.0
                * (k + 1)
            )
        )

        c[n_next] = (
            c[n_current]
            * ratio
        )

        k += 1

    return c


def squeezed_vacuum_retained_probability(
    dim,
    r,
    phi=0.0,
    max_n=None,
):
    """
    Probability contained in the retained squeezed-vacuum
    Fock components BEFORE numerical renormalization.
    """
    c = (
        squeezed_vacuum_fock_coefficients(
            dim,
            r,
            phi=phi,
            max_n=max_n,
        )
    )

    return float(
        np.sum(
            np.abs(c) ** 2
        )
    )


def squeezed_vacuum_state(
    dim,
    r,
    phi=0.0,
    max_n=None,
):
    """
    Return normalized finite-Fock representation
    of squeezed vacuum.
    """
    c = (
        squeezed_vacuum_fock_coefficients(
            dim,
            r,
            phi=phi,
            max_n=max_n,
        )
    )

    return (
        _normalized_ket_from_coefficients(
            c
        )
    )


# ============================================================
# |0> / |1> MIXTURE
# ============================================================

def vacuum_single_photon_mixture(
    dim,
    p_one,
):
    """
    Return

        rho = (1-p)|0><0| + p|1><1|.

    This will later provide the finite transition:

        p > 1/2
            Wigner negativity present

        p <= 1/2
            Wigner function nonnegative

    while the state remains non-Gaussian for

        0 < p < 1.
    """
    dim = _require_dimension(
        dim
    )

    if dim < 2:
        raise ValueError(
            "Mixture requires dim >= 2."
        )

    p_one = float(
        p_one
    )

    if not (
        0.0 <= p_one <= 1.0
    ):
        raise ValueError(
            "p_one must satisfy 0 <= p_one <= 1."
        )

    ket0 = basis(
        dim,
        0,
    )

    ket1 = basis(
        dim,
        1,
    )

    rho = (
        (1.0 - p_one)
        * ket2dm(ket0)
        +
        p_one
        * ket2dm(ket1)
    )

    return rho


# ============================================================
# SIMPLE STATE UTILITIES
# ============================================================

def density_matrix(
    state,
):
    """
    Convert ket -> density matrix.

    Density matrices are returned unchanged.
    """
    if state.isket:

        return ket2dm(
            state
        )

    if state.isoper:

        return state

    raise TypeError(
        "state must be a ket or density operator."
    )


def state_purity(
    state,
):
    """
    Return Tr(rho^2).
    """
    rho = density_matrix(
        state
    )

    return float(
        np.real(
            (rho * rho).tr()
        )
    )


def fock_probabilities(
    state,
):
    """
    Return diagonal Fock probabilities P_n.
    """
    rho = density_matrix(
        state
    )

    # Explicit copy is required because np.diag() may
    # return a read-only view depending on NumPy version.
    p = np.real(
        np.diag(
            rho.full()
        )
    ).copy()

    # Remove tiny numerical roundoff.
    p[
        np.abs(p) < 1.0e-15
    ] = 0.0

    return p


# ============================================================
# CELL 1 TEST HELPERS
# ============================================================

def _cell1_close(
    value,
    reference,
    atol=1.0e-12,
    rtol=1.0e-10,
):
    """
    Scalar numerical comparison helper.
    """
    return bool(
        np.isclose(
            value,
            reference,
            atol=atol,
            rtol=rtol,
        )
    )


# ============================================================
# CELL 1 TESTS
# ============================================================

def run_gaussian_state_cell1_tests():

    tests = []

    max_error = 0.0


    # --------------------------------------------------------
    # TEST 1
    # Fock state support and normalization
    # --------------------------------------------------------

    dim = 12
    n = 4

    ket = fock_state(
        dim,
        n,
    )

    p = fock_probabilities(
        ket
    )

    expected = np.zeros(
        dim,
        dtype=float,
    )

    expected[n] = 1.0

    err = max(
        abs(
            ket.norm()
            - 1.0
        ),
        np.max(
            np.abs(
                p
                - expected
            )
        ),
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Fock support / normalization",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 2
    # Coherent-state recurrence
    #
    # c_(n+1) / c_n =
    #
    #     alpha / sqrt(n+1)
    # --------------------------------------------------------

    dim = 20
    max_n = 8
    alpha = 0.7 + 0.2j

    c = coherent_fock_coefficients(
        dim,
        alpha,
        max_n=max_n,
    )

    err = 0.0

    for n in range(max_n):

        expected = (
            alpha
            / np.sqrt(n + 1)
        )

        obtained = (
            c[n + 1]
            / c[n]
        )

        err = max(
            err,
            abs(
                obtained
                - expected
            ),
        )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Coherent coefficient recurrence",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 3
    # Coherent retained probability
    # vs analytic Poisson sum
    # --------------------------------------------------------

    dim = 20
    max_n = 6
    alpha = 1.1

    mu = (
        abs(alpha) ** 2
    )

    expected = sum(

        np.exp(-mu)
        * mu**n
        / math.factorial(n)

        for n in range(
            max_n + 1
        )
    )

    obtained = (
        coherent_retained_probability(
            dim,
            alpha,
            max_n=max_n,
        )
    )

    err = abs(
        obtained
        - expected
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Coherent retained probability",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 4
    # Coherent numerical state normalization
    # --------------------------------------------------------

    ket = coherent_state(
        30,
        1.3 + 0.4j,
    )

    err = abs(
        ket.norm()
        - 1.0
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Coherent state normalization",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 5
    # Squeezed vacuum:
    # odd Fock states must vanish
    # --------------------------------------------------------

    c = (
        squeezed_vacuum_fock_coefficients(
            30,
            r=0.7,
            phi=0.3,
        )
    )

    odd_error = np.max(
        np.abs(
            c[1::2]
        )
    )

    max_error = max(
        max_error,
        float(odd_error),
    )

    tests.append(
        (
            "Squeezed vacuum even parity",
            odd_error < 1.0e-14,
        )
    )


    # --------------------------------------------------------
    # TEST 6
    # Squeezed-vacuum recurrence
    # --------------------------------------------------------

    dim = 30
    r = 0.6
    phi = 0.25

    c = (
        squeezed_vacuum_fock_coefficients(
            dim,
            r,
            phi=phi,
            max_n=12,
        )
    )

    factor = (
        -np.exp(1j * phi)
        * np.tanh(r)
    )

    err = 0.0

    for k in range(6):

        n = 2 * k

        expected_ratio = (
            factor
            * np.sqrt(
                (2 * k + 2)
                * (2 * k + 1)
            )
            / (
                2.0
                * (k + 1)
            )
        )

        obtained_ratio = (
            c[n + 2]
            / c[n]
        )

        err = max(
            err,
            abs(
                obtained_ratio
                - expected_ratio
            ),
        )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Squeezed coefficient recurrence",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 7
    # Squeezed vacuum numerical state normalization
    # --------------------------------------------------------

    ket = squeezed_vacuum_state(
        40,
        r=0.8,
        phi=0.4,
    )

    err = abs(
        ket.norm()
        - 1.0
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Squeezed state normalization",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 8
    # |0>/<1> mixture:
    #
    # diagonal populations and purity
    #
    # purity =
    #
    #     (1-p)^2 + p^2
    # --------------------------------------------------------

    p_one = 0.30

    rho = (
        vacuum_single_photon_mixture(
            10,
            p_one,
        )
    )

    probs = fock_probabilities(
        rho
    )

    expected_probs = np.zeros(
        10,
        dtype=float,
    )

    expected_probs[0] = (
        1.0
        - p_one
    )

    expected_probs[1] = (
        p_one
    )

    expected_purity = (
        (1.0 - p_one) ** 2
        + p_one**2
    )

    err = max(
        abs(
            complex(
                rho.tr()
            ).real
            - 1.0
        ),
        np.max(
            np.abs(
                probs
                - expected_probs
            )
        ),
        abs(
            state_purity(rho)
            - expected_purity
        ),
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Mixture populations / purity",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 9
    # Mixture endpoints
    # --------------------------------------------------------

    rho0 = (
        vacuum_single_photon_mixture(
            8,
            0.0,
        )
    )

    rho1 = (
        vacuum_single_photon_mixture(
            8,
            1.0,
        )
    )

    expected0 = ket2dm(
        basis(
            8,
            0,
        )
    )

    expected1 = ket2dm(
        basis(
            8,
            1,
        )
    )

    err = max(
        np.max(
            np.abs(
                rho0.full()
                - expected0.full()
            )
        ),
        np.max(
            np.abs(
                rho1.full()
                - expected1.full()
            )
        ),
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Mixture endpoints",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 10
    # Invalid input handling
    # --------------------------------------------------------

    invalid_ok = True

    try:

        fock_state(
            5,
            5,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        coherent_state(
            5,
            1.0,
            max_n=5,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        squeezed_vacuum_state(
            10,
            r=-0.1,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        vacuum_single_photon_mixture(
            10,
            1.2,
        )

        invalid_ok = False

    except ValueError:
        pass


    tests.append(
        (
            "Invalid input handling",
            invalid_ok,
        )
    )


    # ========================================================
    # REPORT
    # ========================================================

    passed = sum(
        int(ok)
        for _, ok in tests
    )

    total = len(
        tests
    )


    print(
        "=" * 72
    )

    print(
        "GAUSSIAN / NON-GAUSSIAN SIMULATOR"
    )

    print(
        "CELL 1 TESTS"
    )

    print(
        "=" * 72
    )


    for i, (
        name,
        ok,
    ) in enumerate(
        tests,
        start=1,
    ):

        status = (
            "PASS"
            if ok
            else "FAIL"
        )

        print(
            f"{i:2d}. "
            f"{name:<40s} "
            f"{status}"
        )


    print(
        "-" * 72
    )

    print(
        f"Tests passed : "
        f"{passed}/{total}"
    )

    print(
        f"Max error    : "
        f"{max_error:.3e}"
    )

    print(
        "=" * 72
    )


    if passed != total:

        raise AssertionError(
            "Cell 1 validation failed."
        )


    return {

        "passed":
            passed,

        "total":
            total,

        "max_error":
            max_error,
    }


# ============================================================
# RUN CELL 1 VALIDATION
# ============================================================

CELL1_GAUSSIAN_VALID = False

CELL1_GAUSSIAN_RESULTS = (
    run_gaussian_state_cell1_tests()
)

CELL1_GAUSSIAN_VALID = True

GAUSSIAN / NON-GAUSSIAN SIMULATOR
CELL 1 TESTS
 1. Fock support / normalization             PASS
 2. Coherent coefficient recurrence          PASS
 3. Coherent retained probability            PASS
 4. Coherent state normalization             PASS
 5. Squeezed vacuum even parity              PASS
 6. Squeezed coefficient recurrence          PASS
 7. Squeezed state normalization             PASS
 8. Mixture populations / purity             PASS
 9. Mixture endpoints                        PASS
10. Invalid input handling                   PASS
------------------------------------------------------------------------
Tests passed : 10/10
Max error    : 2.220e-16


In [31]:
# ============================================================
# CELL 2
# Quadrature moments and Gaussianity diagnostics
#
# Requires validated Cell 1.
#
# Physics convention
# ------------------
#
#     X = (a + a^dagger)/sqrt(2)
#
#     P = (a - a^dagger)/(i sqrt(2))
#
# therefore
#
#     [X,P] = i
#
# and the vacuum variances are
#
#     Var(X) = Var(P) = 1/2.
#
#
# Rotated quadrature:
#
#     X_theta = X cos(theta) + P sin(theta)
#
#
# Central moments:
#
#     mu_n(theta)
#       = <(X_theta - <X_theta>)^n>
#
#
# For a Gaussian distribution:
#
#     mu_(2k+1) = 0
#
# and
#
#     mu_(2k)
#       = (2k-1)!! * mu_2^k.
#
#
# We therefore define normalized residuals:
#
# odd n:
#
#     delta_n
#       = mu_n / mu_2^(n/2)
#
# even n:
#
#     delta_n
#       = mu_n /
#         [(n-1)!! mu_2^(n/2)]
#         - 1
#
#
# An exact Gaussian gives
#
#     delta_n = 0
#
# for every tested order and every quadrature direction.
#
#
# IMPORTANT
# ---------
# A finite set of moments is a NUMERICAL GAUSSIANITY
# DIAGNOSTIC, not a formal mathematical proof.
#
# The simulator reports:
#
#     Gaussian state : YES / NO
#
# based on
#
#     max |delta_n(theta)| < tolerance.
#
# ============================================================


import math
import numpy as np

from qutip import (
    destroy,
    expect,
    qeye,
)


# ============================================================
# REQUIRE CELL 1
# ============================================================

if globals().get(
    "CELL1_GAUSSIAN_VALID",
    False,
) is not True:

    raise RuntimeError(
        "Validated Cell 1 must be executed first."
    )


_REQUIRED_CELL1_OBJECTS = [

    "density_matrix",

    "fock_state",

    "coherent_state",

    "squeezed_vacuum_state",

    "vacuum_single_photon_mixture",
]


_missing_cell1 = [

    name

    for name in _REQUIRED_CELL1_OBJECTS

    if name not in globals()
]


if _missing_cell1:

    raise RuntimeError(
        "Cell 1 objects missing: "
        f"{_missing_cell1}"
    )


# ============================================================
# BASIC HELPERS
# ============================================================

def _cell2_state_dimension(
    state,
):
    """
    Return Hilbert-space dimension of a ket or operator.
    """
    if not (
        state.isket
        or state.isoper
    ):

        raise TypeError(
            "state must be a ket "
            "or density operator."
        )

    dim = int(
        state.shape[0]
    )

    if dim < 1:

        raise ValueError(
            "Invalid state dimension."
        )

    return dim


def _cell2_real_scalar(
    value,
    imag_tol=1.0e-10,
):
    """
    Convert a numerically real expectation value
    to float.

    Raise if a significant imaginary component appears.
    """
    z = complex(
        value
    )

    if abs(z.imag) > imag_tol:

        raise ValueError(
            "Expected a real quantity but found "
            f"imaginary component {z.imag:.3e}."
        )

    return float(
        z.real
    )


def _double_factorial_odd(
    n,
):
    """
    Return n!! for odd nonnegative integer n.

    Examples:

        1!! = 1
        3!! = 3
        5!! = 15
        7!! = 105
    """
    if not isinstance(
        n,
        (int, np.integer),
    ):

        raise TypeError(
            "n must be an integer."
        )

    n = int(n)

    if n < 1 or n % 2 == 0:

        raise ValueError(
            "n must be a positive odd integer."
        )

    result = 1

    for k in range(
        1,
        n + 1,
        2,
    ):

        result *= k

    return result


# ============================================================
# QUADRATURE OPERATORS
# ============================================================

def quadrature_operators(
    dim,
):
    """
    Return X and P with convention

        X = (a + a^dagger)/sqrt(2)

        P = (a - a^dagger)/(i sqrt(2)).
    """
    if not isinstance(
        dim,
        (int, np.integer),
    ):

        raise TypeError(
            "dim must be an integer."
        )

    dim = int(dim)

    if dim < 2:

        raise ValueError(
            "dim must be >= 2."
        )

    a = destroy(
        dim
    )

    adag = a.dag()

    X = (
        a
        + adag
    ) / np.sqrt(2.0)

    P = (
        a
        - adag
    ) / (
        1j
        * np.sqrt(2.0)
    )

    return (
        X,
        P,
    )


def rotated_quadrature(
    dim,
    theta,
):
    """
    Return

        X_theta
          = X cos(theta)
          + P sin(theta).
    """
    theta = float(
        theta
    )

    X, P = quadrature_operators(
        dim
    )

    return (
        np.cos(theta) * X
        +
        np.sin(theta) * P
    )


# ============================================================
# QUADRATURE DIRECTIONS
# ============================================================

def quadrature_angles(
    n_directions,
):
    """
    Return equally spaced quadrature directions
    over the interval

        0 <= theta < pi.

    Examples:

        n=1:
            0

        n=2:
            0, pi/2

        n=4:
            0, pi/4, pi/2, 3pi/4
    """
    if not isinstance(
        n_directions,
        (int, np.integer),
    ):

        raise TypeError(
            "n_directions must be an integer."
        )

    n_directions = int(
        n_directions
    )

    if n_directions < 1:

        raise ValueError(
            "n_directions must be >= 1."
        )

    return np.arange(
        n_directions,
        dtype=float,
    ) * (
        np.pi
        / n_directions
    )


# ============================================================
# OPERATOR POWER
# ============================================================

def _operator_power(
    operator,
    order,
):
    """
    Explicit operator power.

    Avoids relying on implementation details of Qobj ** n.
    """
    if not isinstance(
        order,
        (int, np.integer),
    ):

        raise TypeError(
            "order must be an integer."
        )

    order = int(
        order
    )

    if order < 0:

        raise ValueError(
            "order must be >= 0."
        )

    dim = int(
        operator.shape[0]
    )

    result = qeye(
        dim
    )

    for _ in range(order):

        result = (
            result
            * operator
        )

    return result


# ============================================================
# QUADRATURE MEAN AND CENTRAL MOMENTS
# ============================================================

def quadrature_mean(
    state,
    theta,
):
    """
    Return <X_theta>.
    """
    rho = density_matrix(
        state
    )

    dim = _cell2_state_dimension(
        rho
    )

    Xtheta = rotated_quadrature(
        dim,
        theta,
    )

    return _cell2_real_scalar(
        expect(
            Xtheta,
            rho,
        )
    )


def quadrature_central_moment(
    state,
    theta,
    order,
):
    """
    Return central quadrature moment

        mu_order(theta)
          = <(X_theta - <X_theta>)^order>.
    """
    if not isinstance(
        order,
        (int, np.integer),
    ):

        raise TypeError(
            "order must be an integer."
        )

    order = int(
        order
    )

    if order < 1:

        raise ValueError(
            "order must be >= 1."
        )

    rho = density_matrix(
        state
    )

    dim = _cell2_state_dimension(
        rho
    )

    Xtheta = rotated_quadrature(
        dim,
        theta,
    )

    mean = _cell2_real_scalar(
        expect(
            Xtheta,
            rho,
        )
    )

    centered = (
        Xtheta
        - mean * qeye(dim)
    )

    power = _operator_power(
        centered,
        order,
    )

    moment = expect(
        power,
        rho,
    )

    return _cell2_real_scalar(
        moment
    )


def quadrature_central_moments(
    state,
    theta,
    max_order=8,
):
    """
    Return central moments mu_2 ... mu_max_order.
    """
    if not isinstance(
        max_order,
        (int, np.integer),
    ):

        raise TypeError(
            "max_order must be an integer."
        )

    max_order = int(
        max_order
    )

    if max_order < 3:

        raise ValueError(
            "max_order must be >= 3."
        )

    moments = {}

    for order in range(
        2,
        max_order + 1,
    ):

        moments[
            order
        ] = (
            quadrature_central_moment(
                state,
                theta,
                order,
            )
        )

    return moments


# ============================================================
# GAUSSIAN MOMENT EXPECTATION
# ============================================================

def gaussian_expected_central_moment(
    variance,
    order,
):
    """
    Expected central moment for a Gaussian
    with variance mu_2 = variance.

    Odd orders:
        0

    Even orders:
        (order-1)!! * variance^(order/2)
    """
    variance = float(
        variance
    )

    if variance <= 0.0:

        raise ValueError(
            "variance must be > 0."
        )

    if not isinstance(
        order,
        (int, np.integer),
    ):

        raise TypeError(
            "order must be an integer."
        )

    order = int(
        order
    )

    if order < 1:

        raise ValueError(
            "order must be >= 1."
        )

    if order % 2 == 1:

        return 0.0

    coefficient = (
        _double_factorial_odd(
            order - 1
        )
    )

    return float(
        coefficient
        * variance ** (
            order / 2.0
        )
    )


# ============================================================
# NORMALIZED GAUSSIAN RESIDUAL
# ============================================================

def gaussian_moment_residual(
    moment,
    variance,
    order,
):
    """
    Return normalized deviation from Gaussian expectation.

    Odd order:

        delta_n
          = mu_n / mu_2^(n/2)

    Even order:

        delta_n
          = mu_n /
            [(n-1)!! mu_2^(n/2)]
            - 1

    Gaussian target:

        delta_n = 0.
    """
    moment = float(
        moment
    )

    variance = float(
        variance
    )

    if variance <= 0.0:

        raise ValueError(
            "variance must be > 0."
        )

    if not isinstance(
        order,
        (int, np.integer),
    ):

        raise TypeError(
            "order must be an integer."
        )

    order = int(
        order
    )

    if order < 3:

        raise ValueError(
            "Gaussian residual requires order >= 3."
        )

    scale = (
        variance
        ** (
            order / 2.0
        )
    )

    if order % 2 == 1:

        return float(
            moment
            / scale
        )

    coefficient = (
        _double_factorial_odd(
            order - 1
        )
    )

    expected = (
        coefficient
        * scale
    )

    return float(
        moment
        / expected
        - 1.0
    )


# ============================================================
# ONE-DIRECTION DIAGNOSTIC
# ============================================================

def gaussian_moment_residuals_one_direction(
    state,
    theta,
    max_order=8,
):
    """
    Return Gaussian residuals for one quadrature direction.
    """
    moments = (
        quadrature_central_moments(
            state,
            theta,
            max_order=max_order,
        )
    )

    variance = moments[
        2
    ]

    if variance <= 0.0:

        raise ValueError(
            "Quadrature variance must be positive."
        )

    residuals = {}

    for order in range(
        3,
        max_order + 1,
    ):

        residuals[
            order
        ] = (
            gaussian_moment_residual(
                moments[
                    order
                ],
                variance,
                order,
            )
        )

    return {

        "theta":
            float(theta),

        "variance":
            float(variance),

        "moments":
            moments,

        "residuals":
            residuals,
    }


# ============================================================
# MULTI-DIRECTION GAUSSIAN DIAGNOSTIC
# ============================================================

def gaussian_moment_diagnostics(
    state,
    n_directions=4,
    max_order=8,
    tolerance=1.0e-6,
):
    """
    Evaluate Gaussian moment residuals over multiple
    quadrature directions.

    Returns arrays suitable for the later marker plot:

        horizontal axis:
            moment order

        vertical axis:
            residual

        one marker per quadrature direction.
    """
    if tolerance <= 0.0:

        raise ValueError(
            "tolerance must be > 0."
        )

    if not isinstance(
        max_order,
        (int, np.integer),
    ):

        raise TypeError(
            "max_order must be an integer."
        )

    max_order = int(
        max_order
    )

    if max_order < 3:

        raise ValueError(
            "max_order must be >= 3."
        )

    angles = quadrature_angles(
        n_directions
    )

    orders = np.arange(
        3,
        max_order + 1,
        dtype=int,
    )

    residual_matrix = np.zeros(
        (
            len(angles),
            len(orders),
        ),
        dtype=float,
    )

    variance_array = np.zeros(
        len(angles),
        dtype=float,
    )

    direction_results = []


    for i, theta in enumerate(
        angles
    ):

        result = (
            gaussian_moment_residuals_one_direction(
                state,
                theta,
                max_order=max_order,
            )
        )

        variance_array[
            i
        ] = result[
            "variance"
        ]

        for j, order in enumerate(
            orders
        ):

            residual_matrix[
                i,
                j,
            ] = result[
                "residuals"
            ][
                int(order)
            ]

        direction_results.append(
            result
        )


    max_abs_residual = float(
        np.max(
            np.abs(
                residual_matrix
            )
        )
    )


    gaussian_consistent = bool(
        max_abs_residual
        < tolerance
    )


    worst_index = np.unravel_index(
        np.argmax(
            np.abs(
                residual_matrix
            )
        ),
        residual_matrix.shape,
    )


    worst_direction_index = int(
        worst_index[0]
    )

    worst_order_index = int(
        worst_index[1]
    )


    worst_theta = float(
        angles[
            worst_direction_index
        ]
    )

    worst_order = int(
        orders[
            worst_order_index
        ]
    )

    worst_residual = float(
        residual_matrix[
            worst_direction_index,
            worst_order_index,
        ]
    )


    return {

        "angles":
            angles,

        "orders":
            orders,

        "residuals":
            residual_matrix,

        "variances":
            variance_array,

        "direction_results":
            direction_results,

        "max_abs_residual":
            max_abs_residual,

        "gaussian":
            gaussian_consistent,

        "tolerance":
            float(tolerance),

        "worst_theta":
            worst_theta,

        "worst_order":
            worst_order,

        "worst_residual":
            worst_residual,
    }


# ============================================================
# COMPACT GAUSSIAN CLASSIFIER
# ============================================================

def classify_gaussian_from_moments(
    state,
    n_directions=4,
    max_order=8,
    tolerance=1.0e-6,
):
    """
    Compact Gaussian classifier.

    Returns the same diagnostic information but provides
    an explicit display string for the later UI.
    """
    result = gaussian_moment_diagnostics(

        state,

        n_directions=
            n_directions,

        max_order=
            max_order,

        tolerance=
            tolerance,
    )

    label = (
        "YES"
        if result[
            "gaussian"
        ]
        else
        "NO"
    )

    result[
        "label"
    ] = label

    result[
        "display"
    ] = (
        f"Gaussian state : {label}  "
        f"(max violation = "
        f"{result['max_abs_residual']:.3e})"
    )

    return result


# ============================================================
# CELL 2 TESTS
# ============================================================

def run_gaussian_state_cell2_tests():

    tests = []

    max_error = 0.0


    # --------------------------------------------------------
    # TEST 1
    # Vacuum mean and variance
    #
    #     <X_theta> = 0
    #
    #     Var(X_theta) = 1/2
    #
    # for all theta.
    # --------------------------------------------------------

    vacuum = fock_state(
        20,
        0,
    )

    angles = quadrature_angles(
        8
    )

    err = 0.0

    for theta in angles:

        mean = quadrature_mean(
            vacuum,
            theta,
        )

        variance = (
            quadrature_central_moment(
                vacuum,
                theta,
                2,
            )
        )

        err = max(
            err,
            abs(mean),
            abs(
                variance
                - 0.5
            ),
        )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Vacuum mean / variance",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 2
    # Coherent-state quadrature mean and variance
    #
    #     <X_theta>
    #       = sqrt(2) Re[
    #           alpha exp(-i theta)
    #         ]
    #
    #     Var = 1/2
    # --------------------------------------------------------

    alpha = (
        0.55
        + 0.25j
    )

    coherent = coherent_state(
        50,
        alpha,
    )

    angles = quadrature_angles(
        6
    )

    err = 0.0

    for theta in angles:

        mean = quadrature_mean(
            coherent,
            theta,
        )

        expected_mean = (
            np.sqrt(2.0)
            * np.real(
                alpha
                * np.exp(
                    -1j
                    * theta
                )
            )
        )

        variance = (
            quadrature_central_moment(
                coherent,
                theta,
                2,
            )
        )

        err = max(
            err,
            abs(
                mean
                - expected_mean
            ),
            abs(
                variance
                - 0.5
            ),
        )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Coherent mean / variance",
            err < 1.0e-10,
        )
    )


    # --------------------------------------------------------
    # TEST 3
    # Squeezed-vacuum quadrature variance
    #
    # Convention from Cell 1:
    #
    # Var(X_theta)
    #
    #   = 1/2 [
    #       cosh(2r)
    #       - sinh(2r) cos(2theta - phi)
    #     ]
    # --------------------------------------------------------

    r = 0.35
    phi = 0.40

    squeezed = squeezed_vacuum_state(
        70,
        r,
        phi=phi,
    )

    angles = quadrature_angles(
        8
    )

    err = 0.0

    for theta in angles:

        obtained = (
            quadrature_central_moment(
                squeezed,
                theta,
                2,
            )
        )

        expected = (
            0.5
            * (
                np.cosh(
                    2.0 * r
                )
                -
                np.sinh(
                    2.0 * r
                )
                * np.cos(
                    2.0 * theta
                    - phi
                )
            )
        )

        err = max(
            err,
            abs(
                obtained
                - expected
            ),
        )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Squeezed analytic variance",
            err < 1.0e-10,
        )
    )


    # --------------------------------------------------------
    # TEST 4
    # Vacuum Gaussian residuals
    # --------------------------------------------------------

    result = (
        gaussian_moment_diagnostics(
            vacuum,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-10,
        )
    )

    err = result[
        "max_abs_residual"
    ]

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Vacuum Gaussian residuals",
            (
                result[
                    "gaussian"
                ]
                and err < 1.0e-12
            ),
        )
    )


    # --------------------------------------------------------
    # TEST 5
    # Coherent Gaussian residuals
    # --------------------------------------------------------

    result = (
        gaussian_moment_diagnostics(
            coherent,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    err = result[
        "max_abs_residual"
    ]

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Coherent Gaussian residuals",
            (
                result[
                    "gaussian"
                ]
                and err < 1.0e-7
            ),
        )
    )


    # --------------------------------------------------------
    # TEST 6
    # Squeezed Gaussian residuals
    # --------------------------------------------------------

    result = (
        gaussian_moment_diagnostics(
            squeezed,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    err = result[
        "max_abs_residual"
    ]

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Squeezed Gaussian residuals",
            (
                result[
                    "gaussian"
                ]
                and err < 1.0e-7
            ),
        )
    )


    # --------------------------------------------------------
    # TEST 7
    # |1> analytic fourth-moment residual
    #
    # For n=1 Fock state:
    #
    #     mu_2 = 3/2
    #
    #     mu_4 = 15/4
    #
    # Therefore
    #
    #     delta_4
    #       = (15/4)/(3*(3/2)^2) - 1
    #
    #       = -4/9.
    #
    # Odd moments vanish by parity.
    # --------------------------------------------------------

    one = fock_state(
        20,
        1,
    )

    result = (
        gaussian_moment_residuals_one_direction(
            one,
            theta=0.37,
            max_order=8,
        )
    )

    obtained_delta4 = (
        result[
            "residuals"
        ][4]
    )

    expected_delta4 = (
        -4.0
        / 9.0
    )

    odd_error = max(
        abs(
            result[
                "residuals"
            ][3]
        ),
        abs(
            result[
                "residuals"
            ][5]
        ),
        abs(
            result[
                "residuals"
            ][7]
        ),
    )

    err = max(
        abs(
            obtained_delta4
            - expected_delta4
        ),
        odd_error,
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Fock |1> analytic residual",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 8
    # Positive-Wigner mixture precursor:
    #
    # rho =
    #   0.75 |0><0|
    # + 0.25 |1><1|
    #
    # Its Wigner positivity will be tested in Cell 3.
    #
    # Here we test that it remains non-Gaussian.
    #
    # For p = 1/4:
    #
    #     mu_2 = 1/2 + p = 3/4
    #
    #     mu_4 = 3/4 + 3p = 3/2
    #
    # giving
    #
    #     delta_4 = -1/9.
    # --------------------------------------------------------

    p_one = 0.25

    mixture = (
        vacuum_single_photon_mixture(
            20,
            p_one,
        )
    )

    result = (
        gaussian_moment_residuals_one_direction(
            mixture,
            theta=0.23,
            max_order=8,
        )
    )

    expected_delta4 = (
        -1.0
        / 9.0
    )

    obtained_delta4 = (
        result[
            "residuals"
        ][4]
    )

    err = abs(
        obtained_delta4
        - expected_delta4
    )

    max_error = max(
        max_error,
        float(err),
    )

    tests.append(
        (
            "Mixture analytic non-Gaussian residual",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 9
    # Direction generation
    #
    # n=4:
    #
    #     0, pi/4, pi/2, 3pi/4
    # --------------------------------------------------------

    obtained = quadrature_angles(
        4
    )

    expected = np.array(
        [
            0.0,
            np.pi / 4.0,
            np.pi / 2.0,
            3.0 * np.pi / 4.0,
        ]
    )

    err = float(
        np.max(
            np.abs(
                obtained
                - expected
            )
        )
    )

    max_error = max(
        max_error,
        err,
    )

    tests.append(
        (
            "Quadrature direction generation",
            err < 1.0e-15,
        )
    )


    # --------------------------------------------------------
    # TEST 10
    # Classifier:
    #
    # coherent -> Gaussian
    #
    # squeezed -> Gaussian
    #
    # Fock |1> -> non-Gaussian
    #
    # mixture -> non-Gaussian
    # --------------------------------------------------------

    coh_class = (
        classify_gaussian_from_moments(
            coherent,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    sq_class = (
        classify_gaussian_from_moments(
            squeezed,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    fock_class = (
        classify_gaussian_from_moments(
            one,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    mix_class = (
        classify_gaussian_from_moments(
            mixture,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    classifier_ok = (

        coh_class[
            "gaussian"
        ]

        and

        sq_class[
            "gaussian"
        ]

        and

        not fock_class[
            "gaussian"
        ]

        and

        not mix_class[
            "gaussian"
        ]
    )

    tests.append(
        (
            "Gaussian classifier",
            classifier_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 11
    # Invalid inputs
    # --------------------------------------------------------

    invalid_ok = True


    try:

        quadrature_angles(
            0
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        quadrature_operators(
            1
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        gaussian_moment_residual(
            1.0,
            0.5,
            2,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        gaussian_moment_diagnostics(
            vacuum,
            n_directions=4,
            max_order=2,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        gaussian_moment_diagnostics(
            vacuum,
            tolerance=0.0,
        )

        invalid_ok = False

    except ValueError:
        pass


    tests.append(
        (
            "Invalid input handling",
            invalid_ok,
        )
    )


    # ========================================================
    # REPORT
    # ========================================================

    passed = sum(
        int(ok)
        for _, ok in tests
    )

    total = len(
        tests
    )


    print(
        "=" * 72
    )

    print(
        "GAUSSIAN / NON-GAUSSIAN SIMULATOR"
    )

    print(
        "CELL 2 TESTS"
    )

    print(
        "=" * 72
    )


    for i, (
        name,
        ok,
    ) in enumerate(
        tests,
        start=1,
    ):

        status = (
            "PASS"
            if ok
            else "FAIL"
        )

        print(
            f"{i:2d}. "
            f"{name:<43s} "
            f"{status}"
        )


    print(
        "-" * 72
    )

    print(
        f"Tests passed       : "
        f"{passed}/{total}"
    )

    print(
        f"Max analytic error : "
        f"{max_error:.3e}"
    )


    # Useful diagnostic fingerprints.

    coh_diag = (
        classify_gaussian_from_moments(
            coherent,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    sq_diag = (
        classify_gaussian_from_moments(
            squeezed,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    fock_diag = (
        classify_gaussian_from_moments(
            one,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )

    mix_diag = (
        classify_gaussian_from_moments(
            mixture,
            n_directions=8,
            max_order=8,
            tolerance=1.0e-7,
        )
    )


    print(
        "-"
        * 72
    )

    print(
        "Gaussian residual fingerprints"
    )

    print(
        f"  coherent : "
        f"{coh_diag['max_abs_residual']:.3e}"
    )

    print(
        f"  squeezed : "
        f"{sq_diag['max_abs_residual']:.3e}"
    )

    print(
        f"  Fock |1> : "
        f"{fock_diag['max_abs_residual']:.3e}"
    )

    print(
        f"  mixture  : "
        f"{mix_diag['max_abs_residual']:.3e}"
    )

    print(
        "=" * 72
    )


    if passed != total:

        raise AssertionError(
            "Cell 2 validation failed."
        )


    return {

        "passed":
            passed,

        "total":
            total,

        "max_error":
            max_error,

        "coherent_max_residual":
            coh_diag[
                "max_abs_residual"
            ],

        "squeezed_max_residual":
            sq_diag[
                "max_abs_residual"
            ],

        "fock1_max_residual":
            fock_diag[
                "max_abs_residual"
            ],

        "mixture_max_residual":
            mix_diag[
                "max_abs_residual"
            ],
    }


# ============================================================
# RUN CELL 2 VALIDATION
# ============================================================

CELL2_GAUSSIAN_VALID = False

CELL2_GAUSSIAN_RESULTS = (
    run_gaussian_state_cell2_tests()
)

CELL2_GAUSSIAN_VALID = True

GAUSSIAN / NON-GAUSSIAN SIMULATOR
CELL 2 TESTS
 1. Vacuum mean / variance                      PASS
 2. Coherent mean / variance                    PASS
 3. Squeezed analytic variance                  PASS
 4. Vacuum Gaussian residuals                   PASS
 5. Coherent Gaussian residuals                 PASS
 6. Squeezed Gaussian residuals                 PASS
 7. Fock |1> analytic residual                  PASS
 8. Mixture analytic non-Gaussian residual      PASS
 9. Quadrature direction generation             PASS
10. Gaussian classifier                         PASS
11. Invalid input handling                      PASS
------------------------------------------------------------------------
Tests passed       : 11/11
Max analytic error : 6.289e-13
------------------------------------------------------------------------
Gaussian residual fingerprints
  coherent : 6.289e-13
  squeezed : 8.860e-14
  Fock |1> : 8.889e-01
  mixture  : 4.074e-01


In [32]:
# ============================================================
# CELL 3
# Wigner functions, negativity, and combined classifier
#
# Requires validated Cells 1 and 2.
#
# IMPORTANT NUMERICAL POLICY
# --------------------------
# Ideal coherent and squeezed states have nonnegative Wigner
# functions. Their finite-Fock representations can acquire
# extremely small negative ripples because finite pure
# truncations are not exact Gaussian states.
#
# We therefore define a RESOLVED Wigner-negativity threshold:
#
#     DEFAULT_WIGNER_TOLERANCE = 1e-7
#
# Values W >= -tolerance are treated as numerically
# nonnegative, while the raw minimum W is always reported.
# ============================================================

import numpy as np
from qutip import wigner


# ============================================================
# REQUIRE CELLS 1 AND 2
# ============================================================

if globals().get("CELL1_GAUSSIAN_VALID", False) is not True:
    raise RuntimeError("Validated Cell 1 must be executed first.")

if globals().get("CELL2_GAUSSIAN_VALID", False) is not True:
    raise RuntimeError("Validated Cell 2 must be executed first.")


_REQUIRED_CELL3_OBJECTS = [
    "density_matrix",
    "state_purity",
    "fock_state",
    "coherent_state",
    "squeezed_vacuum_state",
    "vacuum_single_photon_mixture",
    "classify_gaussian_from_moments",
]

_missing = [
    name for name in _REQUIRED_CELL3_OBJECTS
    if name not in globals()
]

if _missing:
    raise RuntimeError(
        f"Required earlier-cell objects missing: {_missing}"
    )


# ============================================================
# NUMERICAL POLICY
# ============================================================

DEFAULT_WIGNER_TOLERANCE = 1.0e-7


# ============================================================
# PHASE-SPACE GRID
# ============================================================

def phase_space_grid(extent=5.0, n_points=201):

    extent = float(extent)

    if extent <= 0.0:
        raise ValueError("extent must be > 0.")

    if not isinstance(n_points, (int, np.integer)):
        raise TypeError("n_points must be an integer.")

    n_points = int(n_points)

    if n_points < 11:
        raise ValueError("n_points must be >= 11.")

    if n_points % 2 == 0:
        raise ValueError(
            "n_points must be odd so the grid contains zero."
        )

    xvec = np.linspace(-extent, extent, n_points)
    pvec = np.linspace(-extent, extent, n_points)

    return xvec, pvec


# ============================================================
# NUMERICAL INTEGRATION
# ============================================================

def _cell3_trapezoid(values, coordinates, axis):

    if hasattr(np, "trapezoid"):
        return np.trapezoid(
            values,
            coordinates,
            axis=axis,
        )

    return np.trapz(
        values,
        coordinates,
        axis=axis,
    )


def integrate_phase_space(values, xvec, pvec):

    values = np.asarray(values, dtype=float)
    xvec = np.asarray(xvec, dtype=float)
    pvec = np.asarray(pvec, dtype=float)

    expected_shape = (
        len(pvec),
        len(xvec),
    )

    if values.shape != expected_shape:
        raise ValueError(
            f"values must have shape {expected_shape}, "
            f"found {values.shape}."
        )

    integral_x = _cell3_trapezoid(
        values,
        xvec,
        axis=1,
    )

    return float(
        _cell3_trapezoid(
            integral_x,
            pvec,
            axis=0,
        )
    )


# ============================================================
# ANALYTIC WIGNER BENCHMARKS
# ============================================================

def analytic_wigner_vacuum(xvec, pvec):

    X, P = np.meshgrid(
        np.asarray(xvec, dtype=float),
        np.asarray(pvec, dtype=float),
        indexing="xy",
    )

    r2 = X**2 + P**2

    return np.exp(-r2) / np.pi


def analytic_wigner_fock1(xvec, pvec):

    X, P = np.meshgrid(
        np.asarray(xvec, dtype=float),
        np.asarray(pvec, dtype=float),
        indexing="xy",
    )

    r2 = X**2 + P**2

    return (
        (2.0 * r2 - 1.0)
        * np.exp(-r2)
        / np.pi
    )


def analytic_wigner_vacuum_single_photon_mixture(
    xvec,
    pvec,
    p_one,
):

    p_one = float(p_one)

    if not 0.0 <= p_one <= 1.0:
        raise ValueError(
            "p_one must satisfy 0 <= p_one <= 1."
        )

    X, P = np.meshgrid(
        np.asarray(xvec, dtype=float),
        np.asarray(pvec, dtype=float),
        indexing="xy",
    )

    r2 = X**2 + P**2

    return (
        np.exp(-r2)
        / np.pi
        * (
            1.0
            - 2.0 * p_one
            + 2.0 * p_one * r2
        )
    )


# ============================================================
# NUMERICAL WIGNER
# ============================================================

def compute_wigner(state, xvec, pvec):

    rho = density_matrix(state)

    xvec = np.asarray(xvec, dtype=float)
    pvec = np.asarray(pvec, dtype=float)

    if xvec.ndim != 1:
        raise ValueError("xvec must be one-dimensional.")

    if pvec.ndim != 1:
        raise ValueError("pvec must be one-dimensional.")

    W = wigner(
        rho,
        xvec,
        pvec,
        g=np.sqrt(2.0),
    )

    W = np.asarray(W, dtype=float)

    expected_shape = (
        len(pvec),
        len(xvec),
    )

    if W.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected Wigner shape {W.shape}; "
            f"expected {expected_shape}."
        )

    return W


# ============================================================
# WIGNER MINIMUM
# ============================================================

def wigner_minimum(W, xvec, pvec):

    W = np.asarray(W, dtype=float)

    index = np.unravel_index(
        np.argmin(W),
        W.shape,
    )

    ip = int(index[0])
    ix = int(index[1])

    return {
        "minimum": float(W[ip, ix]),
        "x": float(xvec[ix]),
        "p": float(pvec[ip]),
        "index": (ip, ix),
    }


# ============================================================
# NEGATIVE VOLUME
# ============================================================

def wigner_negative_volume(W, xvec, pvec):

    negative_part = np.maximum(
        -np.asarray(W, dtype=float),
        0.0,
    )

    return integrate_phase_space(
        negative_part,
        xvec,
        pvec,
    )


# ============================================================
# WIGNER DIAGNOSTICS
# ============================================================

def wigner_diagnostics(
    state,
    xvec,
    pvec,
    positivity_tolerance=DEFAULT_WIGNER_TOLERANCE,
):

    positivity_tolerance = float(
        positivity_tolerance
    )

    if positivity_tolerance <= 0.0:
        raise ValueError(
            "positivity_tolerance must be > 0."
        )

    W = compute_wigner(
        state,
        xvec,
        pvec,
    )

    integral = integrate_phase_space(
        W,
        xvec,
        pvec,
    )

    minimum_result = wigner_minimum(
        W,
        xvec,
        pvec,
    )

    min_W = minimum_result["minimum"]

    negative_volume = wigner_negative_volume(
        W,
        xvec,
        pvec,
    )

    all_positive = bool(
        min_W >= -positivity_tolerance
    )

    label = (
        "YES"
        if all_positive
        else "NO"
    )

    return {
        "W": W,
        "integral": float(integral),
        "min_W": float(min_W),
        "min_x": minimum_result["x"],
        "min_p": minimum_result["p"],
        "negative_volume": float(negative_volume),
        "all_positive": all_positive,
        "numerically_nonnegative": all_positive,
        "label": label,
        "positivity_tolerance": positivity_tolerance,
        "display": (
            f"Wigner all positive : {label}  "
            f"(min W = {min_W:.3e}, "
            f"tol = {positivity_tolerance:.1e})"
        ),
    }


# ============================================================
# COMBINED CLASSIFIER
# ============================================================

def classify_state_gaussian_wigner(
    state,
    xvec,
    pvec,
    n_directions=8,
    max_order=8,
    gaussian_tolerance=1.0e-6,
    wigner_tolerance=DEFAULT_WIGNER_TOLERANCE,
):

    gaussian_result = classify_gaussian_from_moments(
        state,
        n_directions=n_directions,
        max_order=max_order,
        tolerance=gaussian_tolerance,
    )

    wigner_result = wigner_diagnostics(
        state,
        xvec,
        pvec,
        positivity_tolerance=wigner_tolerance,
    )

    purity = state_purity(state)

    return {
        "gaussian":
            gaussian_result["gaussian"],

        "gaussian_label":
            gaussian_result["label"],

        "max_moment_violation":
            gaussian_result["max_abs_residual"],

        "gaussian_tolerance":
            gaussian_result["tolerance"],

        "moment_diagnostics":
            gaussian_result,

        "wigner_all_positive":
            wigner_result["all_positive"],

        "wigner_label":
            wigner_result["label"],

        "min_W":
            wigner_result["min_W"],

        "negative_volume":
            wigner_result["negative_volume"],

        "wigner_integral":
            wigner_result["integral"],

        "wigner_tolerance":
            wigner_result["positivity_tolerance"],

        "wigner_diagnostics":
            wigner_result,

        "purity":
            purity,

        "gaussian_display":
            (
                f"Gaussian state      : "
                f"{gaussian_result['label']}  "
                f"(max violation = "
                f"{gaussian_result['max_abs_residual']:.3e})"
            ),

        "wigner_display":
            (
                f"Wigner all positive : "
                f"{wigner_result['label']}  "
                f"(min W = "
                f"{wigner_result['min_W']:.3e}, "
                f"tol = "
                f"{wigner_result['positivity_tolerance']:.1e})"
            ),

        "purity_display":
            f"Purity              : {purity:.6f}",
    }


# ============================================================
# CELL 3 TESTS
# ============================================================

def run_gaussian_state_cell3_tests():

    tests = []
    max_analytic_error = 0.0

    xvec, pvec = phase_space_grid(
        extent=5.0,
        n_points=201,
    )

    center = len(xvec) // 2


    # --------------------------------------------------------
    # TEST 1: grid
    # --------------------------------------------------------

    err = max(
        abs(xvec[center]),
        abs(pvec[center]),
        abs(xvec[0] + xvec[-1]),
        abs(pvec[0] + pvec[-1]),
    )

    max_analytic_error = max(
        max_analytic_error,
        float(err),
    )

    tests.append(
        (
            "Phase-space grid / origin",
            err < 1.0e-15,
        )
    )


    # --------------------------------------------------------
    # TEST 2: vacuum analytic Wigner
    # --------------------------------------------------------

    vacuum = fock_state(30, 0)

    W_numeric = compute_wigner(
        vacuum,
        xvec,
        pvec,
    )

    W_exact = analytic_wigner_vacuum(
        xvec,
        pvec,
    )

    err = float(
        np.max(
            np.abs(
                W_numeric - W_exact
            )
        )
    )

    max_analytic_error = max(
        max_analytic_error,
        err,
    )

    tests.append(
        (
            "Vacuum analytic Wigner",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 3: |1> analytic Wigner
    # --------------------------------------------------------

    one = fock_state(30, 1)

    W_numeric = compute_wigner(
        one,
        xvec,
        pvec,
    )

    W_exact = analytic_wigner_fock1(
        xvec,
        pvec,
    )

    err = float(
        np.max(
            np.abs(
                W_numeric - W_exact
            )
        )
    )

    max_analytic_error = max(
        max_analytic_error,
        err,
    )

    tests.append(
        (
            "Fock |1> analytic Wigner",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 4: normalization / quadratures
    # --------------------------------------------------------

    W0 = compute_wigner(
        vacuum,
        xvec,
        pvec,
    )

    X, P = np.meshgrid(
        xvec,
        pvec,
        indexing="xy",
    )

    norm = integrate_phase_space(
        W0,
        xvec,
        pvec,
    )

    x2 = integrate_phase_space(
        X**2 * W0,
        xvec,
        pvec,
    )

    p2 = integrate_phase_space(
        P**2 * W0,
        xvec,
        pvec,
    )

    err = max(
        abs(norm - 1.0),
        abs(x2 - 0.5),
        abs(p2 - 0.5),
    )

    max_analytic_error = max(
        max_analytic_error,
        float(err),
    )

    tests.append(
        (
            "Wigner normalization / quadratures",
            err < 1.0e-9,
        )
    )


    # --------------------------------------------------------
    # TEST 5: coherent center
    # --------------------------------------------------------

    alpha = 0.65 + 0.30j

    coherent = coherent_state(
        50,
        alpha,
    )

    x_coh, p_coh = phase_space_grid(
        extent=6.0,
        n_points=241,
    )

    Wc = compute_wigner(
        coherent,
        x_coh,
        p_coh,
    )

    Xc, Pc = np.meshgrid(
        x_coh,
        p_coh,
        indexing="xy",
    )

    norm_c = integrate_phase_space(
        Wc,
        x_coh,
        p_coh,
    )

    mean_x = integrate_phase_space(
        Xc * Wc,
        x_coh,
        p_coh,
    )

    mean_p = integrate_phase_space(
        Pc * Wc,
        x_coh,
        p_coh,
    )

    expected_x = (
        np.sqrt(2.0)
        * np.real(alpha)
    )

    expected_p = (
        np.sqrt(2.0)
        * np.imag(alpha)
    )

    err = max(
        abs(norm_c - 1.0),
        abs(mean_x - expected_x),
        abs(mean_p - expected_p),
    )

    max_analytic_error = max(
        max_analytic_error,
        float(err),
    )

    tests.append(
        (
            "Coherent Wigner center",
            err < 1.0e-10,
        )
    )


    # --------------------------------------------------------
    # TEST 6: |1> negative volume
    # --------------------------------------------------------

    xfine, pfine = phase_space_grid(
        extent=5.0,
        n_points=401,
    )

    W1_fine = compute_wigner(
        one,
        xfine,
        pfine,
    )

    obtained_negative_volume = (
        wigner_negative_volume(
            W1_fine,
            xfine,
            pfine,
        )
    )

    expected_negative_volume = (
        2.0 * np.exp(-0.5) - 1.0
    )

    err = abs(
        obtained_negative_volume
        - expected_negative_volume
    )

    max_analytic_error = max(
        max_analytic_error,
        float(err),
    )

    tests.append(
        (
            "Fock |1> negative volume",
            err < 5.0e-4,
        )
    )


    # --------------------------------------------------------
    # TEST 7: mixture W(0,0)
    # --------------------------------------------------------

    err = 0.0

    for p_one in [
        0.0,
        0.25,
        0.50,
        0.75,
        1.0,
    ]:

        rho = (
            vacuum_single_photon_mixture(
                30,
                p_one,
            )
        )

        W = compute_wigner(
            rho,
            xvec,
            pvec,
        )

        obtained = W[
            center,
            center,
        ]

        expected = (
            1.0 - 2.0 * p_one
        ) / np.pi

        err = max(
            err,
            abs(
                obtained - expected
            ),
        )

    max_analytic_error = max(
        max_analytic_error,
        float(err),
    )

    tests.append(
        (
            "Mixture analytic W(0,0)",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 8: exact mixture threshold
    # --------------------------------------------------------

    low = wigner_diagnostics(
        vacuum_single_photon_mixture(
            30,
            0.49,
        ),
        xvec,
        pvec,
    )

    threshold = wigner_diagnostics(
        vacuum_single_photon_mixture(
            30,
            0.50,
        ),
        xvec,
        pvec,
    )

    high = wigner_diagnostics(
        vacuum_single_photon_mixture(
            30,
            0.51,
        ),
        xvec,
        pvec,
    )

    threshold_ok = (
        low["all_positive"]
        and threshold["all_positive"]
        and not high["all_positive"]
    )

    tests.append(
        (
            "Mixture p=1/2 negativity threshold",
            threshold_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 9: ordinary coherent / squeezed positivity
    # --------------------------------------------------------

    squeezed = squeezed_vacuum_state(
        70,
        r=0.35,
        phi=0.40,
    )

    coherent_W = wigner_diagnostics(
        coherent,
        xvec,
        pvec,
    )

    squeezed_W = wigner_diagnostics(
        squeezed,
        xvec,
        pvec,
    )

    family_positive_ok = (
        coherent_W["all_positive"]
        and squeezed_W["all_positive"]
    )

    tests.append(
        (
            "Coherent / squeezed Wigner positive",
            family_positive_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 10
    # Regression: stronger squeezed state
    #
    # Finite Hilbert truncation can produce a tiny raw
    # negative ripple (~1e-8). This must not be classified
    # as resolved physical Wigner negativity.
    # --------------------------------------------------------

    squeezed_r07 = squeezed_vacuum_state(
        60,
        r=0.70,
        phi=0.0,
    )

    squeezed_r07_W = wigner_diagnostics(
        squeezed_r07,
        xvec,
        pvec,
    )

    squeezed_artifact_ok = (
        squeezed_r07_W["all_positive"]
        and
        squeezed_r07_W["min_W"]
        >= -DEFAULT_WIGNER_TOLERANCE
    )

    tests.append(
        (
            "Squeezed truncation-artifact tolerance",
            squeezed_artifact_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 11: combined conceptual classifier
    # --------------------------------------------------------

    mixture = (
        vacuum_single_photon_mixture(
            30,
            0.25,
        )
    )

    coherent_class = (
        classify_state_gaussian_wigner(
            coherent,
            xvec,
            pvec,
            n_directions=8,
            max_order=8,
            gaussian_tolerance=1.0e-7,
        )
    )

    squeezed_class = (
        classify_state_gaussian_wigner(
            squeezed,
            xvec,
            pvec,
            n_directions=8,
            max_order=8,
            gaussian_tolerance=1.0e-7,
        )
    )

    fock_class = (
        classify_state_gaussian_wigner(
            one,
            xvec,
            pvec,
            n_directions=8,
            max_order=8,
            gaussian_tolerance=1.0e-7,
        )
    )

    mixture_class = (
        classify_state_gaussian_wigner(
            mixture,
            xvec,
            pvec,
            n_directions=8,
            max_order=8,
            gaussian_tolerance=1.0e-7,
        )
    )

    classifier_ok = (
        coherent_class["gaussian"]
        and coherent_class["wigner_all_positive"]

        and squeezed_class["gaussian"]
        and squeezed_class["wigner_all_positive"]

        and not fock_class["gaussian"]
        and not fock_class["wigner_all_positive"]

        and not mixture_class["gaussian"]
        and mixture_class["wigner_all_positive"]
    )

    tests.append(
        (
            "Combined state classifier",
            classifier_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 12: invalid inputs
    # --------------------------------------------------------

    invalid_ok = True

    try:
        phase_space_grid(
            extent=0.0,
            n_points=201,
        )
        invalid_ok = False
    except ValueError:
        pass

    try:
        phase_space_grid(
            extent=5.0,
            n_points=200,
        )
        invalid_ok = False
    except ValueError:
        pass

    try:
        wigner_diagnostics(
            vacuum,
            xvec,
            pvec,
            positivity_tolerance=0.0,
        )
        invalid_ok = False
    except ValueError:
        pass

    try:
        integrate_phase_space(
            np.zeros((10, 10)),
            xvec,
            pvec,
        )
        invalid_ok = False
    except ValueError:
        pass

    tests.append(
        (
            "Invalid input handling",
            invalid_ok,
        )
    )


    # ========================================================
    # REPORT
    # ========================================================

    passed = sum(
        int(ok)
        for _, ok in tests
    )

    total = len(tests)

    print("=" * 72)
    print("GAUSSIAN / NON-GAUSSIAN SIMULATOR")
    print("CELL 3 TESTS")
    print("=" * 72)

    for i, (name, ok) in enumerate(
        tests,
        start=1,
    ):

        status = (
            "PASS"
            if ok
            else "FAIL"
        )

        print(
            f"{i:2d}. "
            f"{name:<43s} "
            f"{status}"
        )

    print("-" * 72)

    print(
        f"Tests passed       : "
        f"{passed}/{total}"
    )

    print(
        f"Max analytic error : "
        f"{max_analytic_error:.3e}"
    )

    print(
        f"Wigner tolerance   : "
        f"{DEFAULT_WIGNER_TOLERANCE:.1e}"
    )

    print("-" * 72)

    print("Classifier fingerprints")

    for label, result in [
        ("coherent", coherent_class),
        ("squeezed", squeezed_class),
        ("Fock |1>", fock_class),
        ("mixture p=0.25", mixture_class),
    ]:

        print(f"  {label}:")
        print(
            "    "
            + result["gaussian_display"]
        )
        print(
            "    "
            + result["wigner_display"]
        )
        print(
            "    "
            + result["purity_display"]
        )

    print("-" * 72)

    print(
        "Squeezed r=0.70 raw minimum : "
        f"{squeezed_r07_W['min_W']:.3e}"
    )

    print(
        "Squeezed r=0.70 classification: "
        f"{squeezed_r07_W['label']}"
    )

    print(
        f"|1> negative volume          : "
        f"{obtained_negative_volume:.9f}"
    )

    print(
        f"Exact value                  : "
        f"{expected_negative_volume:.9f}"
    )

    print("=" * 72)

    if passed != total:
        raise AssertionError(
            "Cell 3 validation failed."
        )

    return {
        "passed": passed,
        "total": total,
        "max_analytic_error": max_analytic_error,
        "wigner_tolerance":
            DEFAULT_WIGNER_TOLERANCE,
        "squeezed_r07_min_W":
            squeezed_r07_W["min_W"],
        "fock1_negative_volume":
            obtained_negative_volume,
        "fock1_negative_volume_exact":
            expected_negative_volume,
        "coherent":
            coherent_class,
        "squeezed":
            squeezed_class,
        "fock1":
            fock_class,
        "mixture":
            mixture_class,
    }


# ============================================================
# RUN CELL 3 VALIDATION
# ============================================================

CELL3_GAUSSIAN_VALID = False

CELL3_GAUSSIAN_RESULTS = (
    run_gaussian_state_cell3_tests()
)

CELL3_GAUSSIAN_VALID = True

GAUSSIAN / NON-GAUSSIAN SIMULATOR
CELL 3 TESTS
 1. Phase-space grid / origin                   PASS
 2. Vacuum analytic Wigner                      PASS
 3. Fock |1> analytic Wigner                    PASS
 4. Wigner normalization / quadratures          PASS
 5. Coherent Wigner center                      PASS
 6. Fock |1> negative volume                    PASS
 7. Mixture analytic W(0,0)                     PASS
 8. Mixture p=1/2 negativity threshold          PASS
 9. Coherent / squeezed Wigner positive         PASS
10. Squeezed truncation-artifact tolerance      PASS
11. Combined state classifier                   PASS
12. Invalid input handling                      PASS
------------------------------------------------------------------------
Tests passed       : 12/12
Max analytic error : 9.374e-06
Wigner tolerance   : 1.0e-07
------------------------------------------------------------------------
Classifier fingerprints
  coherent:
    Gaussian state      : YES  (max violation = 

In [33]:
# ============================================================
# CELL 4
# Diagnostic plotting layer
#
# Requires validated Cells 1-3.
#
# NO NEW PHYSICS IS INTRODUCED HERE.
#
# Cell 1 : states
# Cell 2 : quadrature moments / Gaussianity
# Cell 3 : Wigner function / resolved negativity
#
#
# Three-panel display
# -------------------
#
#   1. Wigner function W(x,p)
#   2. Fock probabilities P_n
#   3. Relative Gaussian moment residuals delta_n(theta)
#
#
# USER-CONTROLLED NUMERICAL RESOLUTION
# ------------------------------------
#
# Wigner negativity is classified using
#
#     W_min < -W_cut
#
# where the default cutoff comes from Cell 3:
#
#     DEFAULT_WIGNER_TOLERANCE = 1e-7
#
# The raw minimum W is ALWAYS retained and displayed.
#
#
# MOMENT Y-AXIS
# -------------
#
# moment_ylim = None
#     automatic symmetric range, never smaller than [-1,+1]
#
# moment_ylim = 0.5
#     symmetric range [-0.5,+0.5]
#
# moment_ylim = (-0.2,0.1)
#     arbitrary range
#
# ============================================================


import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# REQUIRE CELLS 1-3
# ============================================================

if globals().get("CELL1_GAUSSIAN_VALID", False) is not True:
    raise RuntimeError(
        "Validated Cell 1 must be executed first."
    )

if globals().get("CELL2_GAUSSIAN_VALID", False) is not True:
    raise RuntimeError(
        "Validated Cell 2 must be executed first."
    )

if globals().get("CELL3_GAUSSIAN_VALID", False) is not True:
    raise RuntimeError(
        "Validated Cell 3 must be executed first."
    )


_REQUIRED_CELL4_OBJECTS = [
    "fock_probabilities",
    "classify_state_gaussian_wigner",
    "phase_space_grid",
    "fock_state",
    "DEFAULT_WIGNER_TOLERANCE",
]


_missing_cell4 = [
    name
    for name in _REQUIRED_CELL4_OBJECTS
    if name not in globals()
]


if _missing_cell4:
    raise RuntimeError(
        "Required earlier-cell objects missing: "
        f"{_missing_cell4}"
    )


# ============================================================
# PREPARE DIAGNOSTIC DATA
# ============================================================

def prepare_state_diagnostic_data(
    state,
    extent=5.0,
    n_points=201,
    n_directions=8,
    max_order=8,
    gaussian_tolerance=1.0e-6,
    wigner_tolerance=DEFAULT_WIGNER_TOLERANCE,
):
    """
    Prepare all previously validated quantities required
    for the diagnostic figure.

    wigner_tolerance is the user-defined resolved-negativity
    cutoff.
    """

    wigner_tolerance = float(
        wigner_tolerance
    )

    if wigner_tolerance <= 0.0:
        raise ValueError(
            "wigner_tolerance must be > 0."
        )


    xvec, pvec = phase_space_grid(
        extent=extent,
        n_points=n_points,
    )


    classification = (
        classify_state_gaussian_wigner(
            state,
            xvec,
            pvec,
            n_directions=n_directions,
            max_order=max_order,
            gaussian_tolerance=gaussian_tolerance,
            wigner_tolerance=wigner_tolerance,
        )
    )


    probabilities = fock_probabilities(
        state
    )


    moment_data = classification[
        "moment_diagnostics"
    ]


    wigner_data = classification[
        "wigner_diagnostics"
    ]


    return {
        "xvec":
            xvec,

        "pvec":
            pvec,

        "W":
            wigner_data["W"],

        "fock_probabilities":
            probabilities,

        "orders":
            moment_data["orders"],

        "angles":
            moment_data["angles"],

        "residuals":
            moment_data["residuals"],

        "classification":
            classification,

        "wigner_tolerance":
            wigner_tolerance,
    }


# ============================================================
# AUTOMATIC FOCK DISPLAY RANGE
# ============================================================

def _cell4_fock_display_max(
    probabilities,
    requested_max=None,
    probability_threshold=1.0e-6,
):
    """
    Determine the highest displayed Fock index.
    """

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    dim = len(
        probabilities
    )


    if requested_max is not None:

        if not isinstance(
            requested_max,
            (int, np.integer),
        ):
            raise TypeError(
                "fock_max_display must be an integer or None."
            )

        requested_max = int(
            requested_max
        )

        if requested_max < 0:
            raise ValueError(
                "fock_max_display must be >= 0."
            )

        return min(
            requested_max,
            dim - 1,
        )


    significant = np.where(
        probabilities > probability_threshold
    )[0]


    if len(significant) == 0:
        highest = 0
    else:
        highest = int(
            significant[-1]
        )


    display_max = max(
        10,
        highest + 2,
    )


    return min(
        display_max,
        dim - 1,
    )


# ============================================================
# MOMENT Y-AXIS RANGE
# ============================================================

def _cell4_moment_ylim(
    residuals,
    moment_ylim=None,
    minimum_auto_range=1.0,
    auto_padding=1.10,
):
    """
    Resolve vertical axis limits for Gaussian residual plot.

    moment_ylim = None:
        automatic symmetric scale, minimum [-1,+1]

    moment_ylim = positive scalar:
        symmetric user scale

    moment_ylim = (ymin,ymax):
        arbitrary user scale
    """

    if minimum_auto_range <= 0.0:
        raise ValueError(
            "minimum_auto_range must be > 0."
        )

    if auto_padding <= 1.0:
        raise ValueError(
            "auto_padding must be > 1."
        )


    # --------------------------------------------------------
    # Automatic
    # --------------------------------------------------------

    if moment_ylim is None:

        residuals = np.asarray(
            residuals,
            dtype=float,
        )

        max_abs = float(
            np.max(
                np.abs(
                    residuals
                )
            )
        )

        ymax = max(
            float(minimum_auto_range),
            auto_padding * max_abs,
        )

        return (
            -ymax,
            ymax,
        )


    # --------------------------------------------------------
    # Scalar -> symmetric
    # --------------------------------------------------------

    if np.isscalar(
        moment_ylim
    ):

        value = float(
            moment_ylim
        )

        if value <= 0.0:
            raise ValueError(
                "Scalar moment_ylim must be > 0."
            )

        return (
            -value,
            value,
        )


    # --------------------------------------------------------
    # Pair -> arbitrary
    # --------------------------------------------------------

    if len(moment_ylim) != 2:
        raise ValueError(
            "moment_ylim must be None, a positive scalar, "
            "or a (ymin,ymax) pair."
        )


    ymin = float(
        moment_ylim[0]
    )

    ymax = float(
        moment_ylim[1]
    )


    if not ymin < ymax:
        raise ValueError(
            "Require moment_ylim[0] < moment_ylim[1]."
        )


    return (
        ymin,
        ymax,
    )


# ============================================================
# CLASSIFIER TEXT
# ============================================================

def diagnostic_classifier_text(
    classification,
):
    """
    Compact classification block.

    Wigner line already contains both raw minimum W
    and numerical cutoff from Cell 3.
    """

    return "\n".join(
        [
            classification["gaussian_display"],
            classification["wigner_display"],
            classification["purity_display"],
        ]
    )


# ============================================================
# MAIN THREE-PANEL PLOT
# ============================================================

def plot_state_diagnostics(
    state,
    state_label="Quantum state",
    extent=5.0,
    n_points=201,
    n_directions=8,
    max_order=8,
    gaussian_tolerance=1.0e-6,
    wigner_tolerance=DEFAULT_WIGNER_TOLERANCE,
    fock_max_display=None,
    moment_jitter=0.08,
    moment_ylim=None,
    figsize=(15.0, 4.8),
):
    """
    Plot:

        Wigner function
        Fock probabilities
        Relative Gaussian moment residuals

    Parameters
    ----------
    wigner_tolerance:
        User-defined numerical Wigner-negativity cutoff.

        A state is treated as resolved Wigner-positive when

            min(W) >= -wigner_tolerance.

        Raw min(W) is still reported.

    moment_ylim:
        None
            automatic symmetric range, minimum [-1,+1]

        positive scalar
            symmetric manual range

        (ymin,ymax)
            arbitrary manual range
    """

    if moment_jitter < 0.0:
        raise ValueError(
            "moment_jitter must be >= 0."
        )


    wigner_tolerance = float(
        wigner_tolerance
    )

    if wigner_tolerance <= 0.0:
        raise ValueError(
            "wigner_tolerance must be > 0."
        )


    data = prepare_state_diagnostic_data(
        state,
        extent=extent,
        n_points=n_points,
        n_directions=n_directions,
        max_order=max_order,
        gaussian_tolerance=gaussian_tolerance,
        wigner_tolerance=wigner_tolerance,
    )


    classification = data[
        "classification"
    ]

    xvec = data[
        "xvec"
    ]

    pvec = data[
        "pvec"
    ]

    W = data[
        "W"
    ]

    probabilities = data[
        "fock_probabilities"
    ]

    orders = data[
        "orders"
    ]

    angles = data[
        "angles"
    ]

    residuals = data[
        "residuals"
    ]


    # ========================================================
    # FIGURE
    # ========================================================

    fig, (
        ax_wigner,
        ax_fock,
        ax_moment,
    ) = plt.subplots(
        1,
        3,
        figsize=figsize,
    )


    # ========================================================
    # PANEL 1
    # WIGNER FUNCTION
    # ========================================================

    max_abs_W = float(
        np.max(
            np.abs(W)
        )
    )


    if max_abs_W <= 0.0:
        max_abs_W = 1.0


    mesh = ax_wigner.pcolormesh(
        xvec,
        pvec,
        W,
        shading="auto",
        cmap="RdBu_r",
        vmin=-max_abs_W,
        vmax=max_abs_W,
    )


    # Draw W=0 contour only when negativity exceeds
    # the selected numerical resolution cutoff.

    if classification[
        "min_W"
    ] < -wigner_tolerance:

        ax_wigner.contour(
            xvec,
            pvec,
            W,
            levels=[0.0],
            linewidths=1.0,
        )


    colorbar = fig.colorbar(
        mesh,
        ax=ax_wigner,
    )


    colorbar.set_label(
        r"$W(x,p)$"
    )


    ax_wigner.set_xlabel(
        r"$x$"
    )

    ax_wigner.set_ylabel(
        r"$p$"
    )

    ax_wigner.set_title(
        "Wigner function"
    )


    ax_wigner.set_aspect(
        "equal",
        adjustable="box",
    )


    # ========================================================
    # PANEL 2
    # FOCK PROBABILITIES
    # ========================================================

    display_max = (
        _cell4_fock_display_max(
            probabilities,
            requested_max=fock_max_display,
        )
    )


    fock_indices = np.arange(
        display_max + 1,
        dtype=int,
    )


    ax_fock.bar(
        fock_indices,
        probabilities[
            :display_max + 1
        ],
    )


    ax_fock.set_xlabel(
        r"Fock state $n$"
    )


    ax_fock.set_ylabel(
        r"$P_n$"
    )


    ax_fock.set_title(
        "Fock probabilities"
    )


    ax_fock.set_xticks(
        fock_indices
    )


    ax_fock.set_ylim(
        bottom=0.0
    )


    ax_fock.grid(
        axis="y",
        alpha=0.25,
    )


    # ========================================================
    # PANEL 3
    # RELATIVE GAUSSIAN MOMENT RESIDUALS
    # ========================================================

    ax_moment.axhline(
        0.0,
        linewidth=1.0,
    )


    n_angle = len(
        angles
    )


    if n_angle == 1:

        offsets = np.array(
            [0.0]
        )

    else:

        offsets = np.linspace(
            -moment_jitter,
            moment_jitter,
            n_angle,
        )


    moment_marker_count = 0


    for i, (
        theta,
        offset,
    ) in enumerate(
        zip(
            angles,
            offsets,
        )
    ):

        theta_deg = (
            theta
            * 180.0
            / np.pi
        )


        x_positions = (
            orders.astype(float)
            + offset
        )


        ax_moment.plot(
            x_positions,
            residuals[i, :],
            linestyle="None",
            marker="+",
            markersize=8,
            markeredgewidth=1.3,
            label=(
                rf"$\theta={theta_deg:.0f}^\circ$"
            ),
        )


        moment_marker_count += len(
            orders
        )


    resolved_moment_ylim = (
        _cell4_moment_ylim(
            residuals,
            moment_ylim=moment_ylim,
        )
    )


    ax_moment.set_ylim(
        resolved_moment_ylim
    )


    ax_moment.set_xlabel(
        "Moment order"
    )


    ax_moment.set_ylabel(
        r"Relative Gaussian moment residual $\delta_n$"
    )


    ax_moment.set_title(
        "Gaussianity violations"
    )


    ax_moment.set_xticks(
        orders
    )


    ax_moment.grid(
        alpha=0.25,
    )


    if n_angle <= 8:

        ax_moment.legend(
            fontsize=8,
            ncol=2,
        )


    # ========================================================
    # FIGURE TITLE + CLASSIFIER
    # ========================================================

    fig.suptitle(
        state_label,
        fontsize=14,
    )


    classifier_text = (
        diagnostic_classifier_text(
            classification
        )
    )


    fig.text(
        0.02,
        0.015,
        classifier_text,
        ha="left",
        va="bottom",
        family="monospace",
        fontsize=9,
    )


    fig.tight_layout(
        rect=(
            0.0,
            0.16,
            1.0,
            0.94,
        )
    )


    axes = {
        "wigner":
            ax_wigner,

        "fock":
            ax_fock,

        "moment":
            ax_moment,

        "colorbar":
            colorbar,

        "moment_marker_count":
            moment_marker_count,

        "moment_ylim":
            resolved_moment_ylim,

        "wigner_tolerance":
            wigner_tolerance,
    }


    return (
        fig,
        axes,
        data,
    )


# ============================================================
# CELL 4 TESTS
# ============================================================

def run_gaussian_state_cell4_tests():

    tests = []

    extent = 4.0
    n_points = 101
    n_directions = 4


    # --------------------------------------------------------
    # REPRESENTATIVE STATES
    # --------------------------------------------------------

    vacuum = fock_state(
        30,
        0,
    )


    coherent = coherent_state(
        40,
        0.6 + 0.2j,
    )


    squeezed = squeezed_vacuum_state(
        60,
        r=0.70,
        phi=0.0,
    )


    fock1 = fock_state(
        30,
        1,
    )


    mixture = (
        vacuum_single_photon_mixture(
            30,
            0.25,
        )
    )


    # --------------------------------------------------------
    # TEST 1
    # Data preparation
    # --------------------------------------------------------

    data = prepare_state_diagnostic_data(
        coherent,
        extent=extent,
        n_points=n_points,
        n_directions=n_directions,
        max_order=8,
    )


    shape_ok = (
        data["W"].shape
        ==
        (
            n_points,
            n_points,
        )

        and

        data["residuals"].shape
        ==
        (
            n_directions,
            6,
        )

        and

        len(
            data["orders"]
        )
        == 6

        and

        abs(
            np.sum(
                data[
                    "fock_probabilities"
                ]
            )
            - 1.0
        )
        < 1.0e-12
    )


    tests.append(
        (
            "Diagnostic data preparation",
            shape_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 2
    # Coherent plot smoke test
    # --------------------------------------------------------

    fig, axes, data = (
        plot_state_diagnostics(
            coherent,
            state_label="Coherent test",
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            max_order=8,
        )
    )


    coherent_plot_ok = (
        fig is not None

        and axes["wigner"] is not None

        and axes["fock"] is not None

        and axes["moment"] is not None

        and data[
            "classification"
        ][
            "gaussian"
        ]

        and data[
            "classification"
        ][
            "wigner_all_positive"
        ]
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Coherent plotting smoke test",
            coherent_plot_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 3
    # Default moment range >= [-1,+1]
    # --------------------------------------------------------

    fig, axes, data = (
        plot_state_diagnostics(
            coherent,
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
        )
    )


    ymin, ymax = axes[
        "moment_ylim"
    ]


    default_range_ok = (
        ymin <= -1.0
        and ymax >= 1.0
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Default moment y-range",
            default_range_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 4
    # Manual symmetric moment range
    # --------------------------------------------------------

    fig, axes, data = (
        plot_state_diagnostics(
            coherent,
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            moment_ylim=0.25,
        )
    )


    symmetric_range_ok = np.allclose(
        axes["moment_ylim"],
        (
            -0.25,
            0.25,
        ),
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Manual symmetric moment y-range",
            symmetric_range_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 5
    # Manual arbitrary moment range
    # --------------------------------------------------------

    fig, axes, data = (
        plot_state_diagnostics(
            coherent,
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            moment_ylim=(-0.2, 0.1),
        )
    )


    arbitrary_range_ok = np.allclose(
        axes["moment_ylim"],
        (
            -0.2,
            0.1,
        ),
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Manual arbitrary moment y-range",
            arbitrary_range_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 6
    # Moment marker count
    # --------------------------------------------------------

    fig, axes, data = (
        plot_state_diagnostics(
            fock1,
            state_label="Fock test",
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            max_order=8,
        )
    )


    expected_markers = (
        n_directions
        * 6
    )


    marker_ok = (
        axes[
            "moment_marker_count"
        ]
        ==
        expected_markers
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Moment marker count",
            marker_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 7
    # Vacuum / representative plotting
    # --------------------------------------------------------

    representative_ok = True


    for state, label in [
        (
            vacuum,
            "Vacuum test",
        ),

        (
            squeezed,
            "Squeezed test",
        ),

        (
            mixture,
            "Mixture test",
        ),
    ]:

        try:

            fig, axes, data = (
                plot_state_diagnostics(
                    state,
                    state_label=label,
                    extent=extent,
                    n_points=n_points,
                    n_directions=n_directions,
                    max_order=8,
                )
            )

            plt.close(
                fig
            )

        except Exception:

            representative_ok = False


    tests.append(
        (
            "Vacuum / representative plotting",
            representative_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 8
    # Positive-Wigner non-Gaussian state
    # --------------------------------------------------------

    data = prepare_state_diagnostic_data(
        mixture,
        extent=extent,
        n_points=n_points,
        n_directions=n_directions,
        max_order=8,
    )


    mixture_classification_ok = (
        not data[
            "classification"
        ][
            "gaussian"
        ]

        and

        data[
            "classification"
        ][
            "wigner_all_positive"
        ]
    )


    tests.append(
        (
            "Positive-Wigner non-Gaussian display",
            mixture_classification_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 9
    # Cell 4 must inherit the Cell-3 default cutoff
    # --------------------------------------------------------

    data = prepare_state_diagnostic_data(
        coherent,
        extent=extent,
        n_points=n_points,
        n_directions=n_directions,
    )


    default_cutoff_ok = np.isclose(
        data[
            "wigner_tolerance"
        ],
        DEFAULT_WIGNER_TOLERANCE,
        rtol=0.0,
        atol=0.0,
    )


    tests.append(
        (
            "Default Wigner cutoff inherited",
            default_cutoff_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 10
    # USER CUTOFF MUST CHANGE RESOLVED NEGATIVITY
    #
    # Choose mixture just above exact threshold:
    #
    #     p = 0.5000001
    #
    # Exact:
    #
    #     W(0,0)
    #       = (1-2p)/pi
    #
    #       ~= -6.37e-8.
    #
    # Therefore:
    #
    #     cutoff 1e-7 -> unresolved -> positive YES
    #
    #     cutoff 1e-8 -> resolved   -> positive NO
    #
    # This is an intentional test of the meaning of
    # numerical Wigner negativity.
    # --------------------------------------------------------

    p_edge = 0.5000001


    edge_state = (
        vacuum_single_photon_mixture(
            30,
            p_edge,
        )
    )


    edge_default = (
        prepare_state_diagnostic_data(
            edge_state,
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            wigner_tolerance=1.0e-7,
        )
    )


    edge_tight = (
        prepare_state_diagnostic_data(
            edge_state,
            extent=extent,
            n_points=n_points,
            n_directions=n_directions,
            wigner_tolerance=1.0e-8,
        )
    )


    raw_min = edge_default[
        "classification"
    ][
        "min_W"
    ]


    expected_min = (
        1.0
        - 2.0 * p_edge
    ) / np.pi


    cutoff_behavior_ok = (
        edge_default[
            "classification"
        ][
            "wigner_all_positive"
        ]

        and

        not edge_tight[
            "classification"
        ][
            "wigner_all_positive"
        ]

        and

        abs(
            raw_min
            - expected_min
        )
        < 1.0e-12
    )


    tests.append(
        (
            "User Wigner cutoff changes resolution",
            cutoff_behavior_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 11
    # Invalid inputs
    # --------------------------------------------------------

    invalid_ok = True


    try:
        plot_state_diagnostics(
            coherent,
            moment_jitter=-0.1,
        )
        invalid_ok = False
    except ValueError:
        pass


    try:
        _cell4_fock_display_max(
            np.array(
                [
                    1.0,
                    0.0,
                ]
            ),
            requested_max=-1,
        )
        invalid_ok = False
    except ValueError:
        pass


    try:
        _cell4_moment_ylim(
            np.zeros(
                (
                    1,
                    6,
                )
            ),
            moment_ylim=0.0,
        )
        invalid_ok = False
    except ValueError:
        pass


    try:
        _cell4_moment_ylim(
            np.zeros(
                (
                    1,
                    6,
                )
            ),
            moment_ylim=(
                1.0,
                -1.0,
            ),
        )
        invalid_ok = False
    except ValueError:
        pass


    try:
        prepare_state_diagnostic_data(
            coherent,
            wigner_tolerance=0.0,
        )
        invalid_ok = False
    except ValueError:
        pass


    try:
        plot_state_diagnostics(
            coherent,
            wigner_tolerance=-1.0e-7,
        )
        invalid_ok = False
    except ValueError:
        pass


    tests.append(
        (
            "Invalid plotting inputs",
            invalid_ok,
        )
    )


    # ========================================================
    # REPORT
    # ========================================================

    passed = sum(
        int(ok)
        for _, ok in tests
    )


    total = len(
        tests
    )


    print(
        "=" * 72
    )

    print(
        "GAUSSIAN / NON-GAUSSIAN SIMULATOR"
    )

    print(
        "CELL 4 TESTS"
    )

    print(
        "=" * 72
    )


    for i, (
        name,
        ok,
    ) in enumerate(
        tests,
        start=1,
    ):

        status = (
            "PASS"
            if ok
            else "FAIL"
        )

        print(
            f"{i:2d}. "
            f"{name:<45s} "
            f"{status}"
        )


    print(
        "-" * 72
    )


    print(
        f"Tests passed          : "
        f"{passed}/{total}"
    )


    print(
        "Physics added         : NONE"
    )


    print(
        "Default moment range  : "
        "at least [-1,+1]"
    )


    print(
        f"Default Wigner cutoff : "
        f"{DEFAULT_WIGNER_TOLERANCE:.1e}"
    )


    print(
        "-" * 72
    )


    print(
        "Cutoff-resolution regression"
    )


    print(
        f"  exact/raw minimum W : "
        f"{raw_min:.3e}"
    )


    print(
        "  cutoff 1e-7         : "
        f"{edge_default['classification']['wigner_label']}"
    )


    print(
        "  cutoff 1e-8         : "
        f"{edge_tight['classification']['wigner_label']}"
    )


    print(
        "=" * 72
    )


    if passed != total:
        raise AssertionError(
            "Cell 4 validation failed."
        )


    return {
        "passed":
            passed,

        "total":
            total,

        "default_wigner_tolerance":
            DEFAULT_WIGNER_TOLERANCE,

        "edge_raw_min":
            raw_min,
    }


# ============================================================
# RUN CELL 4 VALIDATION
# ============================================================

CELL4_GAUSSIAN_VALID = False


CELL4_GAUSSIAN_RESULTS = (
    run_gaussian_state_cell4_tests()
)


CELL4_GAUSSIAN_VALID = True

GAUSSIAN / NON-GAUSSIAN SIMULATOR
CELL 4 TESTS
 1. Diagnostic data preparation                   PASS
 2. Coherent plotting smoke test                  PASS
 3. Default moment y-range                        PASS
 4. Manual symmetric moment y-range               PASS
 5. Manual arbitrary moment y-range               PASS
 6. Moment marker count                           PASS
 7. Vacuum / representative plotting              PASS
 8. Positive-Wigner non-Gaussian display          PASS
 9. Default Wigner cutoff inherited               PASS
10. User Wigner cutoff changes resolution         PASS
11. Invalid plotting inputs                       PASS
------------------------------------------------------------------------
Tests passed          : 11/11
Physics added         : NONE
Default moment range  : at least [-1,+1]
Default Wigner cutoff : 1.0e-07
------------------------------------------------------------------------
Cutoff-resolution regression
  exact/raw minimum W : -6.366e-08
  cuto

In [34]:
# ============================================================
# CELL 5
# Interactive Gaussian / non-Gaussian photonic-state UI
#
# Requires validated Cells 1-4.
#
# NEW UI FEATURE:
#
#     Hilbert-space dimension N
#
# is explicitly user controlled.
#
# This allows the user to perform a genuine basis-convergence
# study rather than having the code silently choose N.
#
# Two distinct numerical questions are therefore visible:
#
#   1. Is the Hilbert-space truncation converged?
#
#   2. Given a converged calculation, what magnitude of
#      Wigner negativity do we regard as numerically resolved?
#
# ============================================================


import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets

from IPython.display import (
    display,
    clear_output,
)


# ============================================================
# REQUIRE CELLS 1-4
# ============================================================

for cell_number in range(1, 5):

    flag = (
        f"CELL{cell_number}_GAUSSIAN_VALID"
    )

    if globals().get(
        flag,
        False,
    ) is not True:

        raise RuntimeError(
            f"Validated Cell {cell_number} "
            "must be executed first."
        )


_REQUIRED_CELL5_OBJECTS = [

    "fock_state",

    "coherent_state",

    "squeezed_vacuum_state",

    "vacuum_single_photon_mixture",

    "prepare_state_diagnostic_data",

    "_cell4_fock_display_max",

    "_cell4_moment_ylim",

    "DEFAULT_WIGNER_TOLERANCE",
]


_missing_cell5 = [

    name

    for name in _REQUIRED_CELL5_OBJECTS

    if name not in globals()
]


if _missing_cell5:

    raise RuntimeError(
        "Required earlier-cell objects missing: "
        f"{_missing_cell5}"
    )


# ============================================================
# UI DEFAULTS
# ============================================================

UI_DEFAULT_HILBERT_DIM = 60

UI_DEFAULT_EXTENT = 5.0

UI_DEFAULT_GRID_POINTS = 151

UI_DEFAULT_GAUSSIAN_TOLERANCE = 1.0e-6


# ============================================================
# SQUEEZING dB <-> r
# ============================================================

def squeezing_db_to_r(
    squeezing_db,
):
    """
    Positive squeezing magnitude in dB:

        S_dB = 10 log10(V_vac / V_sq)

    with

        V_sq / V_vac = exp(-2r)

    therefore

        r = ln(10)/20 * S_dB.
    """

    squeezing_db = float(
        squeezing_db
    )

    if squeezing_db < 0.0:

        raise ValueError(
            "squeezing_db must be >= 0."
        )

    return float(
        np.log(10.0)
        / 20.0
        * squeezing_db
    )


def squeezing_r_to_db(
    r,
):
    """
    Convert r to positive squeezing magnitude in dB.
    """

    r = float(
        r
    )

    if r < 0.0:

        raise ValueError(
            "r must be >= 0."
        )

    return float(
        20.0
        / np.log(10.0)
        * r
    )


# ============================================================
# STATE SELECTION
# ============================================================

def resolve_state_selection(
    selection,
):

    options = {

        "all4": [
            "coherent",
            "squeezed",
            "fock",
            "mixture",
        ],

        "all5": [
            "vacuum",
            "coherent",
            "squeezed",
            "fock",
            "mixture",
        ],

        "vacuum": [
            "vacuum"
        ],

        "coherent": [
            "coherent"
        ],

        "squeezed": [
            "squeezed"
        ],

        "fock": [
            "fock"
        ],

        "mixture": [
            "mixture"
        ],
    }


    if selection not in options:

        raise ValueError(
            f"Unknown state selection: {selection}"
        )


    return list(
        options[
            selection
        ]
    )


# ============================================================
# BUILD STATES FROM UI PARAMETERS
# ============================================================

def build_ui_state_specs(
    selection="all4",
    coherent_amplitude=1.0,
    coherent_phase_deg=0.0,
    squeezing_db=6.0,
    squeezing_phase_deg=0.0,
    fock_n=1,
    mixture_p_one=0.25,
    dim=UI_DEFAULT_HILBERT_DIM,
):
    """
    Construct selected states in a USER-SPECIFIED
    Hilbert-space dimension.
    """

    if not isinstance(
        dim,
        (int, np.integer),
    ):

        raise TypeError(
            "dim must be an integer."
        )


    dim = int(
        dim
    )


    if dim < 2:

        raise ValueError(
            "dim must be >= 2."
        )


    # --------------------------------------------------------
    # Coherent state
    # --------------------------------------------------------

    coherent_amplitude = float(
        coherent_amplitude
    )


    if coherent_amplitude < 0.0:

        raise ValueError(
            "coherent_amplitude must be >= 0."
        )


    coherent_phase_deg = float(
        coherent_phase_deg
    )


    alpha = (
        coherent_amplitude
        * np.exp(
            1j
            * np.deg2rad(
                coherent_phase_deg
            )
        )
    )


    # --------------------------------------------------------
    # Squeezed vacuum
    # --------------------------------------------------------

    squeezing_db = float(
        squeezing_db
    )


    r = squeezing_db_to_r(
        squeezing_db
    )


    squeezing_phase_deg = float(
        squeezing_phase_deg
    )


    squeezing_phase_rad = (
        np.deg2rad(
            squeezing_phase_deg
        )
    )


    # --------------------------------------------------------
    # Fock
    # --------------------------------------------------------

    if not isinstance(
        fock_n,
        (int, np.integer),
    ):

        raise TypeError(
            "fock_n must be an integer."
        )


    fock_n = int(
        fock_n
    )


    if not (
        0 <= fock_n < dim
    ):

        raise ValueError(
            "Require 0 <= fock_n < dim."
        )


    # --------------------------------------------------------
    # Mixture
    # --------------------------------------------------------

    mixture_p_one = float(
        mixture_p_one
    )


    if not (
        0.0
        <= mixture_p_one
        <= 1.0
    ):

        raise ValueError(
            "mixture_p_one must satisfy 0 <= p <= 1."
        )


    # --------------------------------------------------------
    # Construct states
    # --------------------------------------------------------

    all_specs = {

        "vacuum": {

            "key":
                "vacuum",

            "label":
                "Vacuum",

            "state":
                fock_state(
                    dim,
                    0,
                ),
        },


        "coherent": {

            "key":
                "coherent",

            "label":
                (
                    "Coherent"
                    f"   |alpha|={coherent_amplitude:.2f}, "
                    f"phase={coherent_phase_deg:.0f} deg"
                ),

            "state":
                coherent_state(
                    dim,
                    alpha,
                ),
        },


        "squeezed": {

            "key":
                "squeezed",

            "label":
                (
                    "Squeezed vacuum"
                    f"   {squeezing_db:.1f} dB, "
                    f"r={r:.3f}, "
                    f"phase={squeezing_phase_deg:.0f} deg"
                ),

            "state":
                squeezed_vacuum_state(
                    dim,
                    r=r,
                    phi=
                        squeezing_phase_rad,
                ),
        },


        "fock": {

            "key":
                "fock",

            "label":
                (
                    f"Fock state   |{fock_n}>"
                ),

            "state":
                fock_state(
                    dim,
                    fock_n,
                ),
        },


        "mixture": {

            "key":
                "mixture",

            "label":
                (
                    "Mixture"
                    f"   p1={mixture_p_one:.2f}"
                ),

            "state":
                vacuum_single_photon_mixture(
                    dim,
                    mixture_p_one,
                ),
        },
    }


    selected_keys = (
        resolve_state_selection(
            selection
        )
    )


    return [

        all_specs[
            key
        ]

        for key in selected_keys
    ]


# ============================================================
# FOCK TICK HELPER
# ============================================================

def _cell5_fock_ticks(
    display_max,
):

    display_max = int(
        display_max
    )


    if display_max <= 12:

        step = 1

    elif display_max <= 24:

        step = 2

    else:

        step = 5


    return np.arange(
        0,
        display_max + 1,
        step,
        dtype=int,
    )


# ============================================================
# MULTI-STATE COMPARISON FIGURE
# ============================================================

def plot_state_comparison(
    state_specs,
    extent=5.0,
    n_points=151,
    n_directions=8,
    max_order=8,
    gaussian_tolerance=
        UI_DEFAULT_GAUSSIAN_TOLERANCE,
    wigner_tolerance=
        DEFAULT_WIGNER_TOLERANCE,
    moment_ylim=None,
    moment_jitter=0.08,
):
    """
    One state -> one row.

    Multiple states -> multiple rows.

    Columns:

        Wigner
        Fock probabilities
        Gaussianity residuals

    All rows use common plotting scales.
    """

    if len(
        state_specs
    ) < 1:

        raise ValueError(
            "At least one state is required."
        )


    if moment_jitter < 0.0:

        raise ValueError(
            "moment_jitter must be >= 0."
        )


    # ========================================================
    # COMPUTE DIAGNOSTIC DATA
    # ========================================================

    data_list = []


    for spec in state_specs:

        data = (
            prepare_state_diagnostic_data(
                spec[
                    "state"
                ],
                extent=
                    extent,
                n_points=
                    n_points,
                n_directions=
                    n_directions,
                max_order=
                    max_order,
                gaussian_tolerance=
                    gaussian_tolerance,
                wigner_tolerance=
                    wigner_tolerance,
            )
        )

        data_list.append(
            data
        )


    nrows = len(
        state_specs
    )


    # ========================================================
    # COMMON WIGNER SCALE
    # ========================================================

    global_W_abs = max(

        float(
            np.max(
                np.abs(
                    data["W"]
                )
            )
        )

        for data in data_list
    )


    if global_W_abs <= 0.0:

        global_W_abs = 1.0


    # ========================================================
    # COMMON FOCK RANGE
    # ========================================================

    fock_display_max = max(

        _cell4_fock_display_max(
            data[
                "fock_probabilities"
            ]
        )

        for data in data_list
    )


    fock_indices = np.arange(
        fock_display_max + 1,
        dtype=int,
    )


    fock_ticks = _cell5_fock_ticks(
        fock_display_max
    )


    # ========================================================
    # COMMON MOMENT RANGE
    # ========================================================

    all_residuals = np.vstack(
        [
            data[
                "residuals"
            ]
            for data
            in data_list
        ]
    )


    shared_moment_ylim = (
        _cell4_moment_ylim(
            all_residuals,
            moment_ylim=
                moment_ylim,
        )
    )


    # ========================================================
    # FIGURE
    # ========================================================

    fig, axes = plt.subplots(
        nrows,
        3,
        figsize=(
            15.0,
            4.2 * nrows,
        ),
        squeeze=False,
        constrained_layout=True,
    )


    last_mesh = None


    # ========================================================
    # ROW LOOP
    # ========================================================

    for row, (
        spec,
        data,
    ) in enumerate(
        zip(
            state_specs,
            data_list,
        )
    ):

        ax_wigner = axes[
            row,
            0
        ]

        ax_fock = axes[
            row,
            1
        ]

        ax_moment = axes[
            row,
            2
        ]


        classification = data[
            "classification"
        ]


        xvec = data[
            "xvec"
        ]

        pvec = data[
            "pvec"
        ]

        W = data[
            "W"
        ]

        probabilities = data[
            "fock_probabilities"
        ]

        orders = data[
            "orders"
        ]

        angles = data[
            "angles"
        ]

        residuals = data[
            "residuals"
        ]


        # ====================================================
        # WIGNER
        # ====================================================

        last_mesh = (
            ax_wigner.pcolormesh(
                xvec,
                pvec,
                W,
                shading="auto",
                cmap="RdBu_r",
                vmin=
                    -global_W_abs,
                vmax=
                    global_W_abs,
            )
        )


        if (
            classification[
                "min_W"
            ]
            <
            -wigner_tolerance
        ):

            ax_wigner.contour(
                xvec,
                pvec,
                W,
                levels=[
                    0.0
                ],
                linewidths=1.0,
            )


        ax_wigner.set_xlabel(
            r"$x$"
        )

        ax_wigner.set_ylabel(
            r"$p$"
        )

        ax_wigner.set_title(
            spec[
                "label"
            ]
            + "\nWigner function"
        )

        ax_wigner.set_aspect(
            "equal",
            adjustable="box",
        )


        # ====================================================
        # FOCK PROBABILITIES
        # ====================================================

        padded = np.zeros(
            fock_display_max + 1,
            dtype=float,
        )


        available = min(
            len(
                probabilities
            ),
            fock_display_max + 1,
        )


        padded[
            :available
        ] = probabilities[
            :available
        ]


        ax_fock.bar(
            fock_indices,
            padded,
        )


        ax_fock.set_xlim(
            -0.6,
            fock_display_max + 0.6,
        )


        ax_fock.set_ylim(
            0.0,
            1.05,
        )


        ax_fock.set_xticks(
            fock_ticks
        )


        ax_fock.set_xlabel(
            r"Fock state $n$"
        )


        ax_fock.set_ylabel(
            r"$P_n$"
        )


        ax_fock.set_title(
            "Fock probabilities"
        )


        ax_fock.grid(
            axis="y",
            alpha=0.25,
        )


        # ====================================================
        # GAUSSIAN MOMENT RESIDUALS
        # ====================================================

        ax_moment.axhline(
            0.0,
            linewidth=1.0,
        )


        n_angle = len(
            angles
        )


        if n_angle == 1:

            offsets = np.array(
                [
                    0.0
                ]
            )

        else:

            offsets = np.linspace(
                -moment_jitter,
                moment_jitter,
                n_angle,
            )


        for i, (
            theta,
            offset,
        ) in enumerate(
            zip(
                angles,
                offsets,
            )
        ):

            theta_deg = (
                theta
                * 180.0
                / np.pi
            )


            ax_moment.plot(
                orders.astype(float)
                + offset,
                residuals[
                    i,
                    :
                ],
                linestyle="None",
                marker="+",
                markersize=8,
                markeredgewidth=1.3,
                label=(
                    rf"$\theta={theta_deg:.0f}^\circ$"
                ),
            )


        ax_moment.set_ylim(
            shared_moment_ylim
        )


        ax_moment.set_xticks(
            orders
        )


        ax_moment.set_xlabel(
            "Moment order"
        )


        ax_moment.set_ylabel(
            r"Relative Gaussian moment residual $\delta_n$"
        )


        ax_moment.set_title(
            "Gaussianity violations"
        )


        ax_moment.grid(
            alpha=0.25,
        )


        if row == 0:

            ax_moment.legend(
                fontsize=8,
                ncol=2,
            )


    # ========================================================
    # COMMON COLORBAR
    # ========================================================

    wigner_axes = [

        axes[
            row,
            0
        ]

        for row in range(
            nrows
        )
    ]


    colorbar = fig.colorbar(
        last_mesh,
        ax=wigner_axes,
        shrink=0.92,
    )


    colorbar.set_label(
        r"$W(x,p)$"
    )


    fig.suptitle(
        "Gaussian / non-Gaussian photonic-state comparison",
        fontsize=15,
    )


    return (
        fig,
        axes,
        data_list,
        {
            "moment_ylim":
                shared_moment_ylim,

            "wigner_scale":
                (
                    -global_W_abs,
                    global_W_abs,
                ),

            "fock_display_max":
                fock_display_max,

            "colorbar":
                colorbar,
        },
    )


# ============================================================
# TEXT SUMMARY
# ============================================================

def print_comparison_summary(
    state_specs,
    data_list,
    hilbert_dim,
):
    """
    Print numerical classification information.
    """

    print(
        "=" * 78
    )

    print(
        "STATE CLASSIFICATION"
    )

    print(
        "=" * 78
    )


    print(
        f"Hilbert-space dimension N : "
        f"{hilbert_dim}"
    )

    print()


    for spec, data in zip(
        state_specs,
        data_list,
    ):

        c = data[
            "classification"
        ]


        print(
            spec[
                "label"
            ]
        )


        print(
            "  Gaussian state             : "
            f"{c['gaussian_label']} "
            f"(max violation "
            f"{c['max_moment_violation']:.3e})"
        )


        print(
            "  Wigner positive @ cutoff   : "
            f"{c['wigner_label']} "
            f"(min W "
            f"{c['min_W']:.3e}, "
            f"cutoff "
            f"{c['wigner_tolerance']:.1e})"
        )


        print(
            "  Purity                     : "
            f"{c['purity']:.6f}"
        )


        print()


# ============================================================
# BUILD INTERACTIVE UI
# ============================================================

def build_gaussianity_ui():

    style = {
        "description_width":
            "initial"
    }


    # ========================================================
    # STATE SELECTOR
    # ========================================================

    state_selector = widgets.Dropdown(

        options=[
            (
                "All 4 comparison states",
                "all4",
            ),
            (
                "All 5 + vacuum",
                "all5",
            ),
            (
                "Vacuum",
                "vacuum",
            ),
            (
                "Coherent",
                "coherent",
            ),
            (
                "Squeezed vacuum",
                "squeezed",
            ),
            (
                "Fock state",
                "fock",
            ),
            (
                "Vacuum / single-photon mixture",
                "mixture",
            ),
        ],

        value="all4",

        description="State(s)",

        style=style,
    )


    # ========================================================
    # HILBERT-SPACE DIMENSION
    # ========================================================

    hilbert_dim = widgets.IntSlider(

        value=
            UI_DEFAULT_HILBERT_DIM,

        min=20,

        max=400,

        step=10,

        description=
            "Hilbert-space dimension N",

        continuous_update=False,

        style=style,
    )


    hilbert_note = widgets.HTML(

        value=(
            "<i>Basis convergence: increase N until "
            "Wigner minima and moment residuals "
            "no longer change appreciably.</i>"
        )
    )


    # ========================================================
    # COHERENT CONTROLS
    # ========================================================

    coherent_amplitude = widgets.FloatSlider(

        value=1.0,

        min=0.0,

        max=2.5,

        step=0.05,

        description=r"|alpha|",

        continuous_update=False,

        style=style,
    )


    coherent_phase = widgets.FloatSlider(

        value=0.0,

        min=0.0,

        max=360.0,

        step=5.0,

        description="Coherent phase (deg)",

        continuous_update=False,

        style=style,
    )


    coherent_box = widgets.HBox(
        [
            coherent_amplitude,
            coherent_phase,
        ]
    )


    # ========================================================
    # SQUEEZED CONTROLS
    # ========================================================

    squeezing_db = widgets.FloatSlider(

        value=6.0,

        min=0.0,

        max=12.0,

        step=0.1,

        description="Squeezing (dB)",

        continuous_update=False,

        style=style,
    )


    squeezing_phase = widgets.FloatSlider(

        value=0.0,

        min=0.0,

        max=180.0,

        step=5.0,

        description="Squeezing phase (deg)",

        continuous_update=False,

        style=style,
    )


    r_display = widgets.HTML()


    def update_r_display(
        change=None,
    ):

        r_value = squeezing_db_to_r(
            squeezing_db.value
        )

        r_display.value = (
            "<b>Derived:</b> "
            f"r = {r_value:.4f}"
        )


    squeezing_db.observe(
        update_r_display,
        names="value",
    )


    update_r_display()


    squeezed_box = widgets.VBox(
        [
            widgets.HBox(
                [
                    squeezing_db,
                    squeezing_phase,
                ]
            ),
            r_display,
        ]
    )


    # ========================================================
    # FOCK CONTROL
    # ========================================================

    fock_n = widgets.IntSlider(

        value=1,

        min=0,

        max=12,

        step=1,

        description=
            "Fock photon number n",

        continuous_update=False,

        style=style,
    )


    fock_box = widgets.HBox(
        [
            fock_n
        ]
    )


    # ========================================================
    # MIXTURE CONTROL
    # ========================================================

    mixture_p = widgets.FloatSlider(

        value=0.25,

        min=0.0,

        max=1.0,

        step=0.01,

        description="Mixture p1",

        continuous_update=False,

        style=style,
    )


    mixture_box = widgets.HBox(
        [
            mixture_p
        ]
    )


    # ========================================================
    # DIAGNOSTICS
    # ========================================================

    n_directions = widgets.Dropdown(

        options=[
            1,
            2,
            4,
            8,
            12,
        ],

        value=8,

        description=
            "Quadrature directions",

        style=style,
    )


    wigner_cutoff = widgets.FloatLogSlider(

        value=
            DEFAULT_WIGNER_TOLERANCE,

        base=10,

        min=-12,

        max=-4,

        step=0.25,

        description=
            "Wigner negativity cutoff",

        readout_format=".1e",

        continuous_update=False,

        style=style,
    )


    cutoff_note = widgets.HTML(

        value=(
            "<i>Resolved Wigner negativity requires "
            "min(W) &lt; -cutoff. "
            "The raw minimum W is always reported.</i>"
        )
    )


    extent = widgets.FloatSlider(

        value=
            UI_DEFAULT_EXTENT,

        min=3.0,

        max=8.0,

        step=0.5,

        description=
            "Phase-space extent",

        continuous_update=False,

        style=style,
    )


    grid_points = widgets.Dropdown(

        options=[
            101,
            151,
            201,
            251,
        ],

        value=
            UI_DEFAULT_GRID_POINTS,

        description=
            "Wigner grid points",

        style=style,
    )


    # ========================================================
    # MOMENT Y RANGE
    # ========================================================

    moment_auto = widgets.Checkbox(

        value=True,

        description=
            "Automatic moment y-range",
    )


    moment_ymin = widgets.FloatText(

        value=-1.0,

        description="Moment y min",

        disabled=True,

        style=style,
    )


    moment_ymax = widgets.FloatText(

        value=1.0,

        description="Moment y max",

        disabled=True,

        style=style,
    )


    def update_moment_range_controls(
        change=None,
    ):

        manual = (
            not moment_auto.value
        )

        moment_ymin.disabled = (
            not manual
        )

        moment_ymax.disabled = (
            not manual
        )


    moment_auto.observe(
        update_moment_range_controls,
        names="value",
    )


    # ========================================================
    # OUTPUT
    # ========================================================

    output = widgets.Output()


    generate_button = widgets.Button(

        description=
            "Generate / update figure",

        button_style="primary",

        icon="refresh",
    )


    # ========================================================
    # CONTROL VISIBILITY
    # ========================================================

    def refresh_visibility(
        change=None,
    ):

        selection = (
            state_selector.value
        )


        if selection in (
            "all4",
            "all5",
        ):

            visible = {
                "coherent":
                    True,
                "squeezed":
                    True,
                "fock":
                    True,
                "mixture":
                    True,
            }

        else:

            visible = {
                "coherent":
                    selection == "coherent",

                "squeezed":
                    selection == "squeezed",

                "fock":
                    selection == "fock",

                "mixture":
                    selection == "mixture",
            }


        coherent_box.layout.display = (
            ""
            if visible["coherent"]
            else "none"
        )


        squeezed_box.layout.display = (
            ""
            if visible["squeezed"]
            else "none"
        )


        fock_box.layout.display = (
            ""
            if visible["fock"]
            else "none"
        )


        mixture_box.layout.display = (
            ""
            if visible["mixture"]
            else "none"
        )


    state_selector.observe(
        refresh_visibility,
        names="value",
    )


    refresh_visibility()


    # ========================================================
    # RENDER
    # ========================================================

    def render(
        change=None,
    ):

        with output:

            clear_output(
                wait=True
            )


            try:

                # ------------------------------------------------
                # Moment range
                # ------------------------------------------------

                if moment_auto.value:

                    resolved_moment_ylim = None

                else:

                    if not (
                        moment_ymin.value
                        <
                        moment_ymax.value
                    ):

                        raise ValueError(
                            "Moment y min must be "
                            "smaller than moment y max."
                        )


                    resolved_moment_ylim = (
                        float(
                            moment_ymin.value
                        ),
                        float(
                            moment_ymax.value
                        ),
                    )


                # ------------------------------------------------
                # Build states using USER N
                # ------------------------------------------------

                state_specs = (
                    build_ui_state_specs(

                        selection=
                            state_selector.value,

                        coherent_amplitude=
                            coherent_amplitude.value,

                        coherent_phase_deg=
                            coherent_phase.value,

                        squeezing_db=
                            squeezing_db.value,

                        squeezing_phase_deg=
                            squeezing_phase.value,

                        fock_n=
                            fock_n.value,

                        mixture_p_one=
                            mixture_p.value,

                        dim=
                            hilbert_dim.value,
                    )
                )


                # ------------------------------------------------
                # Plot
                # ------------------------------------------------

                fig, axes, data_list, metadata = (
                    plot_state_comparison(

                        state_specs,

                        extent=
                            extent.value,

                        n_points=
                            grid_points.value,

                        n_directions=
                            n_directions.value,

                        max_order=8,

                        gaussian_tolerance=
                            UI_DEFAULT_GAUSSIAN_TOLERANCE,

                        wigner_tolerance=
                            wigner_cutoff.value,

                        moment_ylim=
                            resolved_moment_ylim,
                    )
                )


                print_comparison_summary(
                    state_specs,
                    data_list,
                    hilbert_dim=
                        hilbert_dim.value,
                )


                print(
                    "Shared moment y-range : "
                    f"["
                    f"{metadata['moment_ylim'][0]:.3g}, "
                    f"{metadata['moment_ylim'][1]:.3g}"
                    f"]"
                )


                print(
                    "Wigner cutoff         : "
                    f"{wigner_cutoff.value:.3e}"
                )


                print()


                display(
                    fig
                )


                plt.close(
                    fig
                )


            except Exception as exc:

                print(
                    "UI calculation failed:"
                )

                print(
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )


    generate_button.on_click(
        render
    )


    # ========================================================
    # LAYOUT
    # ========================================================

    ui = widgets.VBox(
        [

            widgets.HTML(
                value=(
                    "<h3>Gaussian / non-Gaussian "
                    "photonic-state simulator</h3>"
                )
            ),

            state_selector,

            widgets.HTML(
                value="<b>Numerical basis</b>"
            ),

            hilbert_dim,

            hilbert_note,

            widgets.HTML(
                value="<b>Coherent state</b>"
            ),

            coherent_box,

            widgets.HTML(
                value="<b>Squeezed vacuum</b>"
            ),

            squeezed_box,

            widgets.HTML(
                value="<b>Fock state</b>"
            ),

            fock_box,

            widgets.HTML(
                value="<b>Mixed state</b>"
            ),

            mixture_box,

            widgets.HTML(
                value="<b>Diagnostics</b>"
            ),

            widgets.HBox(
                [
                    n_directions,
                    wigner_cutoff,
                ]
            ),

            cutoff_note,

            widgets.HBox(
                [
                    extent,
                    grid_points,
                ]
            ),

            widgets.HBox(
                [
                    moment_auto,
                    moment_ymin,
                    moment_ymax,
                ]
            ),

            generate_button,

            output,
        ]
    )


    return {

        "ui":
            ui,

        "output":
            output,

        "render":
            render,

        "controls": {

            "state_selector":
                state_selector,

            "hilbert_dim":
                hilbert_dim,

            "coherent_amplitude":
                coherent_amplitude,

            "coherent_phase":
                coherent_phase,

            "squeezing_db":
                squeezing_db,

            "squeezing_phase":
                squeezing_phase,

            "r_display":
                r_display,

            "fock_n":
                fock_n,

            "mixture_p":
                mixture_p,

            "n_directions":
                n_directions,

            "wigner_cutoff":
                wigner_cutoff,

            "extent":
                extent,

            "grid_points":
                grid_points,

            "moment_auto":
                moment_auto,

            "moment_ymin":
                moment_ymin,

            "moment_ymax":
                moment_ymax,

            "generate_button":
                generate_button,
        },
    }


# ============================================================
# CELL 5 TESTS
# ============================================================

def run_gaussian_state_cell5_tests():

    tests = []

    max_error = 0.0


    # --------------------------------------------------------
    # TEST 1
    # dB -> r
    # --------------------------------------------------------

    db = 6.0

    obtained_r = (
        squeezing_db_to_r(
            db
        )
    )

    expected_r = (
        np.log(10.0)
        / 20.0
        * db
    )

    err = abs(
        obtained_r
        - expected_r
    )

    max_error = max(
        max_error,
        err,
    )

    tests.append(
        (
            "Squeezing dB -> r conversion",
            err < 1.0e-15,
        )
    )


    # --------------------------------------------------------
    # TEST 2
    # dB round trip
    # --------------------------------------------------------

    err = 0.0

    for db_value in [
        0.0,
        3.0,
        6.0,
        8.7,
        10.0,
        12.0,
    ]:

        recovered = (
            squeezing_r_to_db(
                squeezing_db_to_r(
                    db_value
                )
            )
        )

        err = max(
            err,
            abs(
                recovered
                - db_value
            ),
        )


    max_error = max(
        max_error,
        err,
    )


    tests.append(
        (
            "Squeezing dB round trip",
            err < 1.0e-12,
        )
    )


    # --------------------------------------------------------
    # TEST 3
    # Selection
    # --------------------------------------------------------

    selection_ok = (

        resolve_state_selection(
            "all4"
        )
        ==
        [
            "coherent",
            "squeezed",
            "fock",
            "mixture",
        ]

        and

        len(
            resolve_state_selection(
                "all5"
            )
        )
        == 5

        and

        resolve_state_selection(
            "vacuum"
        )
        ==
        [
            "vacuum"
        ]
    )


    tests.append(
        (
            "State pulldown selection logic",
            selection_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 4
    # USER HILBERT DIMENSION PROPAGATION
    # --------------------------------------------------------

    requested_dim = 80


    specs = build_ui_state_specs(

        selection="all4",

        coherent_amplitude=0.7,

        coherent_phase_deg=30.0,

        squeezing_db=6.0,

        squeezing_phase_deg=20.0,

        fock_n=2,

        mixture_p_one=0.30,

        dim=requested_dim,
    )


    dimension_ok = (

        len(
            specs
        )
        == 4

        and

        all(
            spec[
                "state"
            ].shape[0]
            == requested_dim

            for spec in specs
        )
    )


    tests.append(
        (
            "User Hilbert dimension propagation",
            dimension_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 5
    # Single-state one row
    # --------------------------------------------------------

    one_spec = (
        build_ui_state_specs(
            selection="vacuum",
            dim=30,
        )
    )


    fig, axes, data_list, metadata = (
        plot_state_comparison(
            one_spec,
            extent=4.0,
            n_points=61,
            n_directions=4,
        )
    )


    single_row_ok = (

        axes.shape
        ==
        (
            1,
            3,
        )

        and

        len(
            data_list
        )
        == 1

        and

        metadata[
            "moment_ylim"
        ][0]
        <= -1.0

        and

        metadata[
            "moment_ylim"
        ][1]
        >= 1.0
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Single-state one-row figure",
            single_row_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 6
    # All-4 comparison
    # --------------------------------------------------------

    four_specs = (
        build_ui_state_specs(
            selection="all4",
            coherent_amplitude=0.6,
            squeezing_db=4.0,
            fock_n=1,
            mixture_p_one=0.25,
            dim=30,
        )
    )


    fig, axes, data_list, metadata = (
        plot_state_comparison(
            four_specs,
            extent=4.0,
            n_points=61,
            n_directions=4,
            moment_ylim=(
                -1.0,
                1.0,
            ),
        )
    )


    four_row_ok = (

        axes.shape
        ==
        (
            4,
            3,
        )

        and

        len(
            data_list
        )
        == 4

        and

        np.allclose(
            metadata[
                "moment_ylim"
            ],
            (
                -1.0,
                1.0,
            ),
        )
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "All-4 four-row comparison",
            four_row_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 7
    # Wigner cutoff propagation
    # --------------------------------------------------------

    cutoff_test = 1.0e-9


    fig, axes, data_list, metadata = (
        plot_state_comparison(
            one_spec,
            extent=4.0,
            n_points=61,
            n_directions=4,
            wigner_tolerance=
                cutoff_test,
        )
    )


    obtained_cutoff = (
        data_list[
            0
        ][
            "classification"
        ][
            "wigner_tolerance"
        ]
    )


    cutoff_ok = np.isclose(
        obtained_cutoff,
        cutoff_test,
        rtol=0.0,
        atol=0.0,
    )


    plt.close(
        fig
    )


    tests.append(
        (
            "Wigner cutoff propagation",
            cutoff_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 8
    # UI defaults INCLUDING Hilbert dimension
    # --------------------------------------------------------

    ui_test = (
        build_gaussianity_ui()
    )


    controls = ui_test[
        "controls"
    ]


    ui_ok = (

        controls[
            "state_selector"
        ].value
        == "all4"

        and

        controls[
            "hilbert_dim"
        ].value
        ==
        UI_DEFAULT_HILBERT_DIM

        and

        controls[
            "hilbert_dim"
        ].min
        == 20

        and

        controls[
            "hilbert_dim"
        ].max
        == 400

        and

        np.isclose(
            controls[
                "squeezing_db"
            ].value,
            6.0,
        )

        and

        np.isclose(
            controls[
                "wigner_cutoff"
            ].value,
            DEFAULT_WIGNER_TOLERANCE,
        )

        and

        controls[
            "moment_auto"
        ].value
    )


    tests.append(
        (
            "UI construction / defaults",
            ui_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 9
    # Higher user dimension
    # --------------------------------------------------------

    high_dim_specs = (
        build_ui_state_specs(
            selection="squeezed",
            squeezing_db=8.7,
            dim=120,
        )
    )


    high_dim_ok = (

        len(
            high_dim_specs
        )
        == 1

        and

        high_dim_specs[
            0
        ][
            "state"
        ].shape
        ==
        (
            120,
            1,
        )
    )


    tests.append(
        (
            "Higher Hilbert dimension construction",
            high_dim_ok,
        )
    )


    # --------------------------------------------------------
    # TEST 10
    # Invalid inputs
    # --------------------------------------------------------

    invalid_ok = True


    try:

        squeezing_db_to_r(
            -1.0
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        squeezing_r_to_db(
            -0.1
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        resolve_state_selection(
            "not-a-state"
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        build_ui_state_specs(
            dim=1,
        )

        invalid_ok = False

    except ValueError:
        pass


    try:

        build_ui_state_specs(
            fock_n=30,
            dim=30,
        )

        invalid_ok = False

    except ValueError:
        pass


    tests.append(
        (
            "Invalid UI inputs",
            invalid_ok,
        )
    )


    # ========================================================
    # REPORT
    # ========================================================

    passed = sum(
        int(ok)
        for _, ok in tests
    )


    total = len(
        tests
    )


    print(
        "=" * 72
    )


    print(
        "GAUSSIAN / NON-GAUSSIAN SIMULATOR"
    )


    print(
        "CELL 5 TESTS"
    )


    print(
        "=" * 72
    )


    for i, (
        name,
        ok,
    ) in enumerate(
        tests,
        start=1,
    ):

        status = (
            "PASS"
            if ok
            else "FAIL"
        )


        print(
            f"{i:2d}. "
            f"{name:<45s} "
            f"{status}"
        )


    print(
        "-" * 72
    )


    print(
        f"Tests passed       : "
        f"{passed}/{total}"
    )


    print(
        f"Max analytic error : "
        f"{max_error:.3e}"
    )


    print(
        "Physics added      : NONE"
    )


    print(
        f"Default Hilbert N  : "
        f"{UI_DEFAULT_HILBERT_DIM}"
    )


    print(
        f"6 dB -> r          : "
        f"{squeezing_db_to_r(6.0):.6f}"
    )


    print(
        f"Default W cutoff   : "
        f"{DEFAULT_WIGNER_TOLERANCE:.1e}"
    )


    print(
        "=" * 72
    )


    if passed != total:

        raise AssertionError(
            "Cell 5 validation failed."
        )


    return {

        "passed":
            passed,

        "total":
            total,

        "max_error":
            max_error,
    }


# ============================================================
# RUN CELL 5 VALIDATION
# ============================================================

CELL5_GAUSSIAN_VALID = False


CELL5_GAUSSIAN_RESULTS = (
    run_gaussian_state_cell5_tests()
)


CELL5_GAUSSIAN_VALID = True


# ============================================================
# BUILD PRODUCTION UI
# ============================================================

GAUSSIANITY_UI = (
    build_gaussianity_ui()
)


display(
    GAUSSIANITY_UI[
        "ui"
    ]
)


# Initial default rendering.
GAUSSIANITY_UI[
    "render"
]()

GAUSSIAN / NON-GAUSSIAN SIMULATOR
CELL 5 TESTS
 1. Squeezing dB -> r conversion                  PASS
 2. Squeezing dB round trip                       PASS
 3. State pulldown selection logic                PASS
 4. User Hilbert dimension propagation            PASS
 5. Single-state one-row figure                   PASS
 6. All-4 four-row comparison                     PASS
 7. Wigner cutoff propagation                     PASS
 8. UI construction / defaults                    PASS
 9. Higher Hilbert dimension construction         PASS
10. Invalid UI inputs                             PASS
------------------------------------------------------------------------
Tests passed       : 10/10
Max analytic error : 1.776e-15
Physics added      : NONE
Default Hilbert N  : 60
6 dB -> r          : 0.690776
Default W cutoff   : 1.0e-07
